# Regional Style Routing & Baseline Evaluation (Part 1)

This notebook covers the initial experimental pipeline for the dissertation. It handles dataset preparation organising the content images, regional segmentation masks, style references, and text prompts and runs the first phase of comparative evaluations.

### Experimental Overview

To keep compute requirements manageable within single-session GPU limits, the evaluation is divided into two notebooks:

* **Part 1 (Current Notebook):** Implements and evaluates our proposed method (**Method F**) alongside three lightweight baseline approaches:
  * *Sequential regional editing*
  * *Global IP-Adapter conditioning*
  * *IP-Adapter conditioning with rectangular bounding masks*
* **Part 2 (Heavy Baselines):** Focuses on the remaining resource-intensive benchmarks: Neural Style Transfer (NST), AdaIN, InstructPix2Pix, MultiDiffusion, ControlNet, SDXL, and regional attention-bias conditioning.

# FINAL_DISSERTATION_ROUTING_BASELINES

### Qualitative Evaluation: C1/C2 Setup

This notebook evaluates three additional content images under the exact configuration used in the C1/C2 benchmarks. These qualitative samples are assessed independently and are not merged into the 200-sample C1 dataset, leaving existing quantitative results unchanged.

All experimental components are aligned with the C1 pipeline: the base diffusion model, IP-Adapter checkpoints, scheduler parameters, Grounding DINO + SAM segmentation, overlap resolution, style references, prompt templates, and caching logic. Any elements reconstructed in the absence of original code artifacts are explicitly documented in their respective sections.

### Structure

1. Environment and Imports
2. Frozen C1/C2 Configuration Reference
3. Qualitative Sample Definitions (`QUAL_SAMPLES`)
4. Content Image Ingestion
5. Region Definitions
6. C1-Compatible Mask Generation
7. Mask Validation and Overlap Resolution
8. Frozen Style References
9. Prompt Construction
10. Proposed Method (F)
11. Baseline Implementations and Audit Status
12. BrushEdit Integration
13. Generation Pipeline
14. Provenance Tracking
15. Export and Artifact Saving

# Qualitative Baseline Evaluations (C1/C2 Setup)

This notebook evaluates three additional qualitative content images under the exact configuration used in the C1 and C2 experiments. These samples are for visual comparison only and are kept separate from the 200-sample C1 benchmark so existing metrics remain untouched.

All core components mirror the C1 pipeline, including the base diffusion checkpoints, IP-Adapter weights, scheduler parameters, Grounding DINO and SAM segmentation, mask overlap resolution, style references, prompts, and provenance tracking. Any component reconstructed from scratch rather than pulled directly from the original code is noted in its corresponding section.

### Outline

1. Setup and imports
2. C1/C2 configuration reference
3. Qualitative sample selection (`QUAL_SAMPLES`)
4. Image loading and region definitions
5. Mask generation and validation
6. Style references and prompt construction
7. Proposed method (F)
8. Baseline models and BrushEdit integration
9. Image generation
10. Provenance and export

In [1]:
RUN_ID_OVERRIDE = None 

In [2]:
import os
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '5')  
import torch
print(torch.cuda.device_count())
print(torch.cuda.mem_get_info())

1
(8047099904, 8165523456)


## 1. Environment / imports

In [3]:
import os, sys, gc, re, json, glob, time, hashlib, subprocess, random, inspect
import importlib
from datetime import datetime, timezone
from PIL import Image

os.environ.setdefault('PYTHONHASHSEED', '0')

# Keeping new HF dependencies in a separate directory so we aren't breaking existing torch/torchvision setups
_PIN_DIR = os.path.expanduser('~/hf_pin_packages')
os.makedirs(_PIN_DIR, exist_ok=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
    '--target', _PIN_DIR,
    'huggingface_hub>=0.34,<1.0',
    'regex>=2025.10.22',
    'tokenizers>=0.22.0,<=0.23.0',
    'transformers',
    'diffusers==0.31.0'
], check=True)

if _PIN_DIR in sys.path:
    sys.path.remove(_PIN_DIR)
sys.path.insert(0, _PIN_DIR)
importlib.invalidate_caches()

print(f'huggingface_hub/regex/tokenizers/transformers/diffusers pinned in an isolated '
      f'directory, prepended to sys.path: {_PIN_DIR} (torch/torchvision untouched); '
      f'import caches invalidated so this is actually found this session')

import numpy as np
import torch

try:
    import cv2
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python'], check=True)
    import cv2

# Locking down random seeds so runs stay consistent
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

HAS_CUDA = torch.cuda.is_available()
DEVICE = 'cuda' if HAS_CUDA else 'cpu'
DTYPE = torch.float16 if HAS_CUDA else torch.float32

def require_gpu(what):
    if not HAS_CUDA:
        raise RuntimeError(f'{what} needs a GPU. This notebook does not fall back to cached or historical results.')

def vram(tag=''):
    if not HAS_CUDA:
        return None
    free, total = torch.cuda.mem_get_info()
    print(f'    [VRAM] {tag:<28s} free {free/1e9:6.2f} GB / {total/1e9:.2f} GB')
    return free / 1e9

def sha256_file(path, n=64):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()[:n]

def sha256_text(s, n=64):
    return hashlib.sha256(s.encode()).hexdigest()[:n]

# Reusing the existing run ID whenever we are picking up where a previous session left off
RUN_ID = globals().get('RUN_ID_OVERRIDE') or (
    'QUALRUN_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
)

print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if HAS_CUDA else 'no CUDA')
print('run id:', RUN_ID, '(override)' if globals().get('RUN_ID_OVERRIDE') else '(fresh)')
vram('startup')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.15.2 requires numpy<2.5,>=1.23.5, but you have numpy 2.5.3 which is incompatible.
lightning 2.5.1 requires fsspec[http]<2026.0,>=2022.5.0, but you have fsspec 2026.9.0 which is incompatible.
lightning 2.5.1 requires packaging<25.0,>=20.0, but you have packaging 26.3 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


huggingface_hub/regex/tokenizers/transformers/diffusers pinned in an isolated directory, prepended to sys.path: /homes/rkr44/hf_pin_packages (torch/torchvision untouched); import caches invalidated so this is actually found this session
device: cuda | NVIDIA GeForce RTX 2080
run id: QUALRUN_20260919T082948Z (fresh)
    [VRAM] startup                      free   8.05 GB / 8.17 GB


8.047099904

In [25]:
import importlib
import subprocess
import sys

# Installing extra dependencies needed specifically for Grounding DINO and SAM
# Keeping this strictly to segmentation tools since Hugging Face packages are already pinned above to avoid diffuser conflicts

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'addict',
    'yapf',
    'pycocotools',
    'timm',
    'supervision',
    'opencv-python',
    'git+https://github.com/facebookresearch/segment-anything.git'
], check=True)

# Invalidating import caches so the newly installed modules resolve cleanly in this session
importlib.invalidate_caches()

print('Segmentation dependencies installed and import caches refreshed.')

Segmentation dependencies installed and import caches refreshed.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## 2. C1/C2 configuration

Treating C1 and C2 strictly as frozen reference points here. None of their existing artifacts or output directories are being touched or overwritten.

Pointing `C1_RUN_DIR` and `C2_RUN_DIR` to the saved run locations below before proceeding.

In [26]:
# Supporting execution across Google Colab and the university HPC cluster
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    assert os.path.isdir('/content/drive/MyDrive'), (
        'Drive did not mount -- work_dir is a Drive path and needs it. '
        'Re-run this cell and approve the mount prompt.'
    )
    _default_work_dir = '/content/drive/MyDrive/baselines'
    _default_c1_dir = '/content/drive/MyDrive/regional_style_transfer/FINAL_RUN_20260827T132200Z'
    _default_c2_dir = '/content/drive/MyDrive/regional_style_transfer/C2_RUN_20260829T075822Z'
else:
    # Defaulting to local project paths when running on HPC unless environment overrides are provided
    _default_work_dir = os.path.join(os.getcwd(), 'baselines_work')
    _default_c1_dir = os.path.join(os.getcwd(), 'regional_style_transfer', 'FINAL_RUN_20260827T132200Z')
    _default_c2_dir = os.path.join(os.getcwd(), 'regional_style_transfer', 'C2_RUN_20260829T075822Z')

_WORK_DIR = os.environ.get('BASELINES_WORK_DIR', _default_work_dir)

# Pointing to the frozen C1 and C2 runs strictly as read-only references
C1_RUN_DIR = os.environ.get('C1_RUN_DIR', _default_c1_dir)
C2_RUN_DIR = os.environ.get('C2_RUN_DIR', _default_c2_dir)

print('work dir (BASELINES_WORK_DIR):', _WORK_DIR)
print('C1_RUN_DIR:', C1_RUN_DIR)
print('C2_RUN_DIR:', C2_RUN_DIR)

C1_VERIFICATION_PATH = f'{C1_RUN_DIR}/verification/verification.json'
if os.path.exists(C1_VERIFICATION_PATH):
    with open(C1_VERIFICATION_PATH) as f:
        C1_VERIFICATION = json.load(f)

    assert C1_VERIFICATION.get('n_failed_critical', 1) == 0, (
        f'C1 run {C1_RUN_DIR} did not pass its own final verification -- refusing to '
        f'build the proposed-method configuration on an unverified C1.'
    )

    C1_RUN_ID = C1_VERIFICATION.get('run_id')
    C1_CONFIG_HASH = C1_VERIFICATION.get('config_hash')
    print(f'C1 verification loaded: run_id={C1_RUN_ID} config_hash={C1_CONFIG_HASH}')
else:
    print(f'WARNING: {C1_VERIFICATION_PATH} not found in this environment -- C1 provenance '
          f'fields below will be None until C1_RUN_DIR is pointed at a real frozen run. '
          f'The CONFIG values below are still C1\'s frozen values, copied from the C1 '
          f'notebook itself, not re-derived from this missing file.')
    C1_VERIFICATION, C1_RUN_ID, C1_CONFIG_HASH = None, None, None

# Keeping historical metrics from C2 strictly for sanity checks and baseline comparisons
C2_ATTENTION_REFERENCE = {
    'c2_run_id': 'C2_RUN_20260829T075822Z',
    'masked_outside_mask_attention': 0.023077,
    'unmasked_outside_mask_attention': 0.201454,
    'relative_reduction': 0.8854,
    'n_samples_reduced': 200,
    'n_samples_total': 200,
    'n_valid_rows': 400,
    'n_rows_total': 400,
    'hook_errors': 0,
}

# Pulling parameters directly from the validated C1 setup to ensure alignment
CONFIG = {
    'sd_repo': 'stable-diffusion-v1-5/stable-diffusion-inpainting',
    'ip_repo': 'h94/IP-Adapter',
    'ip_weight': 'ip-adapter_sd15.bin',

    'gdino_tag': 'v0.1.0-alpha2',
    'gdino_box_thr': 0.30,
    'gdino_text_thr': 0.25,
    'gdino_retry_thr': 0.15,

    'image_size': (512, 512),
    'seed': 42,
    'steps': 30,
    'guidance': 7.5,
    'strength': 0.75,
    'ip_scale': 0.70,
    'negative_prompt': 'ugly, blurry, distorted, watermark, hard seam',

    'band_px': 20,
    'cond_erode_px': 4,
    'cond_blur_px': 5,
    'cond_ramp_min': 3 / 7,
    'feather_max_alpha': 0.50,
    'crop_context_px': 40,

    'reinpaint_ip_scale': 0.40,
    'reinpaint_steps': 20,

    'work_dir': _WORK_DIR,
}

BAND_PX = CONFIG['band_px']
SEED = CONFIG['seed']

FROZEN_KEYS = [
    'sd_repo', 'ip_repo', 'ip_weight', 'image_size', 'seed', 'steps',
    'guidance', 'strength', 'ip_scale', 'negative_prompt', 'band_px',
    'cond_erode_px', 'cond_blur_px', 'cond_ramp_min', 'feather_max_alpha',
    'reinpaint_ip_scale', 'reinpaint_steps'
]

CONFIG_HASH = hashlib.sha256(
    json.dumps({k: CONFIG[k] for k in FROZEN_KEYS}, sort_keys=True, default=str).encode()
).hexdigest()[:16]

WORK = CONFIG['work_dir']
INPUTS = f'{WORK}/inputs'
WEIGHTS_DIR = f'{WORK}/weights'

# Redirecting model download caches directly into the work directory to avoid bloating home disk quotas
os.environ['TORCH_HOME'] = f'{WORK}/torch_cache'
os.environ['HF_HOME'] = f'{WORK}/hf_cache'

# Isolating outputs per run ID
OUT_DIR = f'{WORK}/outputs/{RUN_ID}'
FIG_DIR = f'{OUT_DIR}/figures'

for d in (INPUTS, WEIGHTS_DIR, OUT_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

print('config hash (this notebook):', CONFIG_HASH)
print('work dir:', WORK)
print('output dir:', OUT_DIR, '(new every run)')

work dir (BASELINES_WORK_DIR): /mnt/vurm/homes/homes/rkr44/baselines_work
C1_RUN_DIR: /mnt/vurm/homes/homes/rkr44/regional_style_transfer/FINAL_RUN_20260827T132200Z
C2_RUN_DIR: /mnt/vurm/homes/homes/rkr44/regional_style_transfer/C2_RUN_20260829T075822Z
config hash (this notebook): 0e99b319cd438079
work dir: /mnt/vurm/homes/homes/rkr44/baselines_work
output dir: /mnt/vurm/homes/homes/rkr44/baselines_work/outputs/QUALRUN_20260919T082948Z (new every run)


## 3. Qualitative Sample Definitions

Selecting three content images for visual evaluation. These images remain unchanged and are not assigned C1 `sample_id` values.

Setting up the target regions, style pairings, and prompt templates using the C1 schema rather than the older routing notebook.

Locking down `QUAL_SAMPLES` here so the definitions remain fixed throughout inference.

In [6]:
QUAL_SAMPLES = [
    {
        'id': 'QUAL_01',
        'label': 'Mountain range above a sea of fog at sunset',
        'content_url': 'https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=512&q=80',
        'region1': 'sky', 'region2': 'mountain',
        'style1_name': 'Van Gogh', 'style2_name': 'Cezanne',
    },
    {
        # Tracking the original content photo here
        # The mountain area is smaller and partially occluded by trees, so checking mask quality downstream
        'id': 'QUAL_02',
        'label': 'Mountain range beyond a forested foreground',
        'content_url': 'https://images.unsplash.com/photo-1464822759023-fed622ff2c3b?w=512&q=80',
        'region1': 'sky', 'region2': 'mountain',
        'style1_name': 'Van Gogh', 'style2_name': 'Cezanne',
    },
    {
        # Updating label from 'misty ridge' to reflect the Milky Way night sky over the silhouette
        'id': 'QUAL_03',
        'label': 'Milky Way over a mountain silhouette at night',
        'content_url': 'https://images.unsplash.com/photo-1519681393784-d120267933ba?w=512&q=80',
        'region1': 'sky', 'region2': 'mountain',
        'style1_name': 'Van Gogh', 'style2_name': 'Cezanne',
    },
]

QUAL_SAMPLE_IDS = [s['id'] for s in QUAL_SAMPLES]

assert QUAL_SAMPLE_IDS == ['QUAL_01', 'QUAL_02', 'QUAL_03'], \
    'Enforcing the three qualitative sample IDs in exact sequence'

# Keeping the sky/mountain pairing and styles consistent across all three images
# Isolating differences in generation quality to the underlying content images
print(f'{len(QUAL_SAMPLES)} qualitative samples defined: {QUAL_SAMPLE_IDS}')

for s in QUAL_SAMPLES:
    print(
        f"  {s['id']}: {s['label']!r}  "
        f"region1={s['region1']} (→{s['style1_name']})  "
        f"region2={s['region2']} (→{s['style2_name']})"
    )

3 qualitative samples defined: ['QUAL_01', 'QUAL_02', 'QUAL_03']
  QUAL_01: 'Mountain range above a sea of fog at sunset'  region1=sky (→Van Gogh)  region2=mountain (→Cezanne)
  QUAL_02: 'Mountain range beyond a forested foreground'  region1=sky (→Van Gogh)  region2=mountain (→Cezanne)
  QUAL_03: 'Milky Way over a mountain silhouette at night'  region1=sky (→Van Gogh)  region2=mountain (→Cezanne)


## 4. Content Loading

Reconstructing `canonical_image` to mirror the original C1 behavior. The function applies a center-square crop, resizes to the target resolution, saves as a lossless PNG with its SHA-256 hash, and reloads from disk.

In [7]:
def _center_square_resize(img, size):
    w, h = img.size
    s = min(w, h)
    return img.crop(((w - s) // 2, (h - s) // 2,
                     (w - s) // 2 + s, (h - s) // 2 + s)).resize(size, Image.LANCZOS)

ASSET_LOG = {}

def canonical_image(source, name, size=None):
    # Reconstructing the C1 loading flow: cropping center square, resizing, and caching as lossless PNG
    size = size or CONFIG['image_size']
    png = f'{INPUTS}/{name}.png'

    if not os.path.exists(png):
        if str(source).startswith('http'):
            import requests
            headers = {'User-Agent': 'Mozilla/5.0 (research script)'}
            r = requests.get(source, timeout=60, headers=headers)
            r.raise_for_status()
            raw = f'{INPUTS}/_raw_{name}'
            with open(raw, 'wb') as f:
                f.write(r.content)
            src = raw
        else:
            if not os.path.exists(source):
                raise FileNotFoundError(f'local file not found: {source}')
            src = source

        _center_square_resize(
            Image.open(src).convert('RGB'), size
        ).save(png, 'PNG')

        ASSET_LOG[name] = {'source': source}
        print('canonicalised', os.path.basename(png))

    img = Image.open(png).convert('RGB')
    img.load()
    return img, sha256_file(png)

# Ingesting and caching qualitative target images locally
QUAL_CONTENT, QUAL_CONTENT_HASH = {}, {}

for s in QUAL_SAMPLES:
    img, h = canonical_image(s['content_url'], f"content_{s['id']}")
    QUAL_CONTENT[s['id']] = img
    QUAL_CONTENT_HASH[s['id']] = h
    print(f"{s['id']}: {img.size} sha256={h}")

canonicalised content_QUAL_01.png
QUAL_01: (512, 512) sha256=3398b74a8520bee2a58f8a016e268b44837a9ef8617c8627f7c4130be699dd0b
canonicalised content_QUAL_02.png
QUAL_02: (512, 512) sha256=47e6d552a6527bbac0fac9b72855eacfa310dac75defb6fb831f2e8d68461a3f
canonicalised content_QUAL_03.png
QUAL_03: (512, 512) sha256=234ef4fa3cbaef36fd1ff9b538afd0509ebc84aa156eb5e8a63c8d9e0dbbfd00


## 5. Region definitions

Using the shared `sky` and `mountain` regions defined in Section 3 across all three samples.Mapping region labels to their canonical descriptions via `REGION_DESCRIPTORS` from C1. These descriptors feed directly into the Grounding DINO text queries in Section 6 and the prompt templates in Section 9.

In [8]:
REGION_DESCRIPTORS = {
    'sky': 'sky', 'mountain': 'mountain landscape', 'mountains': 'mountain landscape',
    'forest': 'forest trees', 'water': 'water reflections', 'valley': 'valley landscape',
    'fog': 'fog and mist', 'undergrowth': 'undergrowth and foliage',
    'buildings': 'buildings and architecture', 'street': 'street and pavement',
    'rocks': 'rocky terrain',
}
for s in QUAL_SAMPLES:
    for r in (s['region1'], s['region2']):
        assert r in REGION_DESCRIPTORS, f"'{r}' has no C1 region descriptor -- add it, don't invent a local one"
print('region descriptors resolved for all samples:',
      {s['id']: (s['region1'], s['region2']) for s in QUAL_SAMPLES})

region descriptors resolved for all samples: {'QUAL_01': ('sky', 'mountain'), 'QUAL_02': ('sky', 'mountain'), 'QUAL_03': ('sky', 'mountain')}


## 6. C1-Compatible Mask Generation

Generating masks via the original `load_segmentation` and `create_masks` implementation from C1. Loading Grounding DINO directly from the pinned `IDEA-Research/GroundingDINO` release alongside Facebook's `segment-anything` repo.Switching away from the `transformers` implementation used in earlier routing experiments to keep mask boundaries and detection thresholds identical to the main C1 benchmark.

In [9]:
MODELS = {}

def load_segmentation():
    # Making sure we actually have access to a GPU before trying to spin up large vision models
    require_gpu('mask generation')
    if 'sam_predictor' in MODELS:
        return

    # Cloning and checking out the pinned Grounding DINO release locally
    gdino_dir = os.path.expanduser('~/GroundingDINO')
    if not os.path.exists(gdino_dir):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/IDEA-Research/GroundingDINO.git', gdino_dir], check=True)
    subprocess.run(['git', '-C', gdino_dir, 'checkout', '-q', CONFIG['gdino_tag']], check=True)

    # Verifying the cloned directory actually contains the required Python package structure
    pkg_init = os.path.join(gdino_dir, 'groundingdino', '__init__.py')
    assert os.path.exists(pkg_init), (
        f'GroundingDINO clone at {gdino_dir} is missing {pkg_init} -- delete {gdino_dir} and re-run.'
    )
    if gdino_dir not in sys.path:
        sys.path.insert(0, gdino_dir)

    from groundingdino.util.inference import load_model
    from segment_anything import sam_model_registry, SamPredictor

    # Setting up weight paths and model configurations on disk
    gd_pth = f'{WEIGHTS_DIR}/gdino.pth'
    gd_cfg = f'{WEIGHTS_DIR}/gdino_config.py'
    sam_pth = f'{WEIGHTS_DIR}/sam_vit_l.pth'

    # Pulling model checkpoints (using SAM ViT-L to avoid OOM issues on T4 instances)
    if not os.path.exists(gd_pth):
        subprocess.run([
            'wget', '-q', '-O', gd_pth,
            'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth'
        ], check=True)

    if not os.path.exists(gd_cfg):
        subprocess.run([
            'wget', '-q', '-O', gd_cfg,
            'https://raw.githubusercontent.com/IDEA-Research/GroundingDINO/main/groundingdino/config/GroundingDINO_SwinT_OGC.py'
        ], check=True)

    if not os.path.exists(sam_pth):
        subprocess.run([
            'wget', '-q', '-O', sam_pth,
            'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth'
        ], check=True)

    # Patching BertModel methods on the fly to bridge newer transformers releases with Grounding DINO
    import transformers as tfm

    # Restoring legacy get_head_mask helper that Grounding DINO internals still rely on
    def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            assert head_mask.dim() == 5, f'head_mask.dim() should be 5, got {head_mask.dim()}'
            head_mask = head_mask.to(dtype=self.dtype)
            if is_attention_chunked:
                head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask

    if not hasattr(tfm.models.bert.modeling_bert.BertModel, 'get_head_mask'):
        tfm.models.bert.modeling_bert.BertModel.get_head_mask = _get_head_mask

    # Matching the exact positional/keyword argument signature expected by the detection pipeline
    def _get_extended_attention_mask(self, attention_mask, input_shape, *args, **kwargs):
        dtype = kwargs.get('dtype', getattr(self, 'dtype', torch.float32))
        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(f'Wrong shape for attention_mask (shape {attention_mask.shape})')
        extended_attention_mask = extended_attention_mask.to(dtype=dtype)
        return (1.0 - extended_attention_mask) * torch.finfo(dtype).min

    tfm.models.bert.modeling_bert.BertModel.get_extended_attention_mask = _get_extended_attention_mask

    # Falling back to pure PyTorch deformable attention when compiled CUDA kernels are unavailable
    import groundingdino.models.GroundingDINO.ms_deform_attn as _msda
    if not hasattr(_msda, '_C'):
        class _Shim:
            @staticmethod
            def ms_deform_attn_forward(v, shapes, lvl, loc, w, step):
                return _msda.multi_scale_deformable_attn_pytorch(v, shapes, loc, w)
        _msda._C = _Shim()

    # Initializing Grounding DINO and SAM predictor into memory
    MODELS['gdino'] = load_model(gd_cfg, gd_pth)
    sam = sam_model_registry['vit_l'](checkpoint=sam_pth).to(DEVICE)
    MODELS['sam'] = sam
    MODELS['sam_predictor'] = SamPredictor(sam)

    print('GroundingDINO and SAM loaded (original repositories, matching C1)')

def free_segmentation():
    # Evicting segmentation weights from VRAM once masks are generated
    for k in ('gdino', 'sam', 'sam_predictor'):
        MODELS.pop(k, None)
    
    # Running explicit garbage collection and clearing CUDA cache to free up memory for the diffusion pipeline
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [27]:
# Reconstructing the mask manipulation helpers needed for create_masks()
# Using morphological dilation for boundary seams and Euclidean distance transforms to split overlaps

def _ellipse(px):
    # Generating an elliptical structuring element for smooth dilation
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (px * 2 + 1, px * 2 + 1))


def boundary_band(m1, m2, band_px=None, max_band_px=60):
    # Dilating both masks to find where their borders intersect
    bp = BAND_PX if band_px is None else band_px
    band = cv2.bitwise_and(
        cv2.dilate(m1, _ellipse(bp)),
        cv2.dilate(m2, _ellipse(bp))
    )

    # Expanding dilation progressively if masks sit close without an immediate boundary overlap
    while band.sum() == 0 and bp < max_band_px:
        bp += 10
        band = cv2.bitwise_and(
            cv2.dilate(m1, _ellipse(bp)),
            cv2.dilate(m2, _ellipse(bp))
        )

    return band


def exterior_band(mask, band_px=None, exclude_mask=None):
    # Expanding the mask outward while subtracting its interior to isolate its perimeter
    bp = BAND_PX if band_px is None else band_px
    band = cv2.subtract(cv2.dilate(mask, _ellipse(bp)), mask)

    # Masking out neighboring segments so the outer band stays within relevant background
    if exclude_mask is not None:
        band = cv2.bitwise_and(band, cv2.bitwise_not(exclude_mask))

    return band


def resolve_mask_overlap(m1, m2):
    # Resolving overlapping regions by measuring Euclidean distance back to each mask's unique core
    overlap = (m1 > 127) & (m2 > 127)

    if not overlap.any():
        return m1.copy(), m2.copy()

    only1 = (m1 > 127) & ~overlap
    only2 = (m2 > 127) & ~overlap
    m1, m2 = m1.copy(), m2.copy()

    # Assigning all disputed pixels directly if one region lacks exclusive territory
    if not only1.any():
        give_to_1 = np.zeros_like(overlap)
    elif not only2.any():
        give_to_1 = np.ones_like(overlap)
    else:
        # Computing L2 distance maps from unambiguous territory to disputed pixels
        bg1 = np.where(only1, 0, 255).astype(np.uint8)
        bg2 = np.where(only2, 0, 255).astype(np.uint8)
        d1 = cv2.distanceTransform(bg1, cv2.DIST_L2, 5)
        d2 = cv2.distanceTransform(bg2, cv2.DIST_L2, 5)
        give_to_1 = d1 <= d2

    # Zeroing out competing pixels based on distance preference
    m1[overlap & ~give_to_1] = 0
    m2[overlap & give_to_1] = 0

    return m1, m2


print('mask helpers defined (from C1), band width =', BAND_PX, 'px')

mask helpers defined (from C1), band width = 20 px


In [11]:
def create_masks(sample_id):
    # Adapting C1 mask generation to target the local QUAL_CONTENT and QUAL_SAMPLES dictionaries
    spec = next(s for s in QUAL_SAMPLES if s['id'] == sample_id)
    content = QUAL_CONTENT[sample_id]
    r1, r2 = spec['region1'], spec['region2']

    p1 = f'{INPUTS}/mask_{sample_id}_{r1}.png'
    p2 = f'{INPUTS}/mask_{sample_id}_{r2}.png'
    raw1 = p1.replace('.png', '_raw.png')
    raw2 = p2.replace('.png', '_raw.png')
    boxes_path = f'{INPUTS}/boxes_{sample_id}.json'

    # Generating detection boxes and segmentations if not already cached to disk
    if not (os.path.exists(raw1) and os.path.exists(raw2) and os.path.exists(boxes_path)):
        load_segmentation()
        import groundingdino.datasets.transforms as T
        from groundingdino.util.inference import predict

        tf = T.Compose([
            T.RandomResize([800], max_size=1333),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
        tensor, _ = tf(content, None)
        Wd, Ht = content.size
        MODELS['sam_predictor'].set_image(np.array(content))

        out, boxes_by_region = [], {}
        for region_name in (r1, r2):
            # Running Grounding DINO to spot bounding boxes matching the text prompt
            boxes, _, _ = predict(
                model=MODELS['gdino'], image=tensor, caption=region_name,
                box_threshold=CONFIG['gdino_box_thr'],
                text_threshold=CONFIG['gdino_text_thr']
            )

            # Retrying with relaxed confidence thresholds if an object region fails detection
            if len(boxes) == 0:
                t = CONFIG['gdino_retry_thr']
                boxes, _, _ = predict(
                    model=MODELS['gdino'], image=tensor, caption=region_name,
                    box_threshold=t, text_threshold=t
                )
            assert len(boxes) > 0, f'GroundingDINO found no "{region_name}" in {sample_id}'

            # Converting normalized center-format boxes to absolute pixel bounds
            px = boxes.clone()
            px[:, 0] = (boxes[:, 0] - boxes[:, 2] / 2) * Wd
            px[:, 1] = (boxes[:, 1] - boxes[:, 3] / 2) * Ht
            px[:, 2] = (boxes[:, 0] + boxes[:, 2] / 2) * Wd
            px[:, 3] = (boxes[:, 1] + boxes[:, 3] / 2) * Ht
            boxes_by_region[region_name] = px.cpu().numpy().tolist()

            # Generating binary segmentation masks per box via SAM and aggregating them
            m = np.zeros((Ht, Wd), np.uint8)
            for box in px.cpu().numpy():
                sm, _, _ = MODELS['sam_predictor'].predict(
                    point_coords=None, point_labels=None,
                    box=np.array(box, np.float32)[None, :], multimask_output=False
                )
                m = np.maximum(m, (sm[0] * 255).astype(np.uint8))
            out.append(m)

        # Caching raw unadjusted masks and detected coordinates
        Image.fromarray(out[0]).save(raw1, 'PNG')
        Image.fromarray(out[1]).save(raw2, 'PNG')
        with open(boxes_path, 'w') as f:
            json.dump(boxes_by_region, f)

    raw_m1 = np.array(Image.open(raw1).convert('L'))
    raw_m2 = np.array(Image.open(raw2).convert('L'))
    with open(boxes_path) as f:
        boxes = json.load(f)

    # Resolving pixel collisions along borders before saving finalized masks
    m1, m2 = resolve_mask_overlap(raw_m1, raw_m2)

    Image.fromarray(m1).save(p1, 'PNG')
    Image.fromarray(m2).save(p2, 'PNG')
    m1 = np.array(Image.open(p1).convert('L'))
    m2 = np.array(Image.open(p2).convert('L'))
    h1, h2 = sha256_file(p1), sha256_file(p2)

    # Validating area coverage, complete partition separation, and seam existence
    a1, a2 = (m1 > 127).mean(), (m2 > 127).mean()
    assert a1 > 0.01 and a2 > 0.01, f'implausibly small mask: {a1:.2%} / {a2:.2%}'
    assert ((m1 > 127) & (m2 > 127)).sum() == 0, 'masks still overlap after resolution'
    assert boundary_band(m1, m2).sum() > 0, 'no seam between the two regions'
    for mask, other, nm in ((m1, m2, r1), (m2, m1, r2)):
        n = (exterior_band(mask, exclude_mask=other) > 127).sum()
        assert n > 0, f'exterior band for "{nm}" is empty'

    return {
        'm1': m1, 'm2': m2, 'h1': h1, 'h2': h2, 'raw_m1': raw_m1, 'raw_m2': raw_m2,
        'boxes': boxes, 'p1': p1, 'p2': p2,
        'raw_overlap_px': int(((raw_m1 > 127) & (raw_m2 > 127)).sum()),
        'coverage_pct': float((np.maximum(m1, m2) > 127).mean() * 100)
    }

# Running mask creation across all qualitative samples
require_gpu('mask generation')
QUAL_MASKS, QUAL_MASK_META = {}, {}
for sid in QUAL_SAMPLE_IDS:
    rec = create_masks(sid)
    QUAL_MASKS[sid] = (rec['m1'], rec['m2'])
    QUAL_MASK_META[sid] = rec
    print(f"{sid}: region1 area={(rec['m1']>127).mean():.1%}  "
          f"region2 area={(rec['m2']>127).mean():.1%}  "
          f"raw_overlap_px={rec['raw_overlap_px']}")

# Dropping segmentation models from VRAM once all masks are processed and verified
free_segmentation()
print('\nsegmentation models released')

/homes/rkr44/.local/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/homes/rkr44/GroundingDINO/groundingdino/models/GroundingDINO/ms_deform_attn.py:31: UserWarning: Failed to load custom C++ ops. Running on CPU mode Only!
  warnings.warn("Failed to load custom C++ ops. Running on CPU mode Only!")
/opt/python/lib/python3.12/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

GroundingDINO and SAM loaded (original repos, matching C1 -- not the transformers ports the old routing notebook used)


/opt/python/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/opt/python/lib/python3.12/site-packages/torch/utils/checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/homes/rkr44/GroundingDINO/groundingdino/models/GroundingDINO/transformer.py:862: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):


QUAL_01: region1 area=39.2%  region2 area=23.4%  raw_overlap_px=0
QUAL_02: region1 area=37.8%  region2 area=31.2%  raw_overlap_px=0
QUAL_03: region1 area=56.6%  region2 area=12.6%  raw_overlap_px=0

segmentation models released


## 7. Mask validation and overlap resolution

Checking the generated masks against their saved hashes and verifying that no overlapping pixels remain between resolved regions. Building a concise provenance table to log coverage and seam metrics across all test images. Any sample failing these validation checks gets flagged immediately and excluded from downstream inference.

In [12]:
import pandas as pd

mask_rows, mask_provenance_failures = [], []

# Iterating through generated masks to verify disk consistency and boundary resolution
for sid, (m1, m2) in QUAL_MASKS.items():
    rec = QUAL_MASK_META[sid]
    spec = next(s for s in QUAL_SAMPLES if s['id'] == sid)
    
    # Reloading saved masks from disk to guarantee they match our in-memory arrays byte-for-byte
    disk_m1 = np.array(Image.open(rec['p1']).convert('L'))
    disk_m2 = np.array(Image.open(rec['p2']).convert('L'))
    same = np.array_equal(disk_m1, m1) and np.array_equal(disk_m2, m2)
    hashes_ok = (sha256_file(rec['p1']) == rec['h1'] and sha256_file(rec['p2']) == rec['h2'])
    overlap = int(((m1 > 127) & (m2 > 127)).sum())
    
    # Flagging any corrupted files, hash mismatches, or lingering border collisions
    if not (same and hashes_ok and overlap == 0):
        mask_provenance_failures.append({
            'sample_id': sid,
            'error': f'in_memory_matches_disk={same} hash_stable={hashes_ok} overlap_px={overlap}'
        })
        
    # Logging provenance attributes and spatial metrics for downstream verification
    mask_rows.append({
        'sample_id': sid, 'region1': spec['region1'], 'region2': spec['region2'],
        'mask1_sha256': rec['h1'], 'mask2_sha256': rec['h2'],
        'region1_area_pct': float((m1 > 127).mean() * 100),
        'region2_area_pct': float((m2 > 127).mean() * 100),
        'raw_overlap_px': rec['raw_overlap_px'], 'resolved_overlap_px': overlap,
        'mask_coverage_pct': rec['coverage_pct'],
        'seam_band_px': int((boundary_band(m1, m2) > 127).sum()),
    })

# Exporting mask metrics table directly to the run output directory
mask_df = pd.DataFrame(mask_rows)
mask_df.to_csv(f'{OUT_DIR}/mask_validation.csv', index=False)

# Pruning failed samples from the active dictionary
for f in mask_provenance_failures:
    QUAL_MASKS.pop(f['sample_id'], None)

# Asserting that all three test items pass strict integrity and zero-overlap requirements
assert not mask_provenance_failures, f'mask provenance failures: {mask_provenance_failures}'
assert (mask_df.resolved_overlap_px == 0).all(), 'a sample still has overlapping masks'
assert set(mask_df.sample_id) == set(QUAL_SAMPLE_IDS), 'not all three qualitative samples produced valid masks'

print(mask_df.to_string(index=False))
print('\nall 3 qualitative samples: masks validated, zero overlap, hashes stable')

sample_id region1  region2                                                     mask1_sha256                                                     mask2_sha256  region1_area_pct  region2_area_pct  raw_overlap_px  resolved_overlap_px  mask_coverage_pct  seam_band_px
  QUAL_01     sky mountain 7740f3b6545acc8b00f11c364b355315f382f1616ec3fa7ab6e7574d57bf46eb 9954df273e556c24d3d46ac5fc4e67a29a60c7873035dd206abc6459840a81ea         39.197159         23.350906               0                    0          62.548065         18200
  QUAL_02     sky mountain bc25ce93f6e3d2010b1977f56fd7946dc097ac9da458fc2750706b690278f064 6f9fd7089f149ace5306e6ece607bba45e37f072ce360bc35213b8407d649efd         37.807846         31.211090               0                    0          69.018936         21086
  QUAL_03     sky mountain 46f00da8328bf0c7e23758db78e1a37cb86201fec1bb19f21446ade801baab36 7e5451111f1da729e157992c318e667e334b19fb05c56160011c9676d34f9450         56.615067         12.583160               0   

## 8. C1 Style References

Reusing the exact Van Gogh and Cezanne style reference images from the C1 run rather than pulling from earlier routing prototypes. Keeping these style exemplars consistent ensures the visual conditioning aligns directly with our primary benchmark experiments.

In [13]:
STYLE_REF_URLS = {
    'Van Gogh': [
        'https://commons.wikimedia.org/wiki/Special:FilePath/Vincent_van_Gogh_-_Wheat_Field_with_Cypresses_-_Google_Art_Project.jpg',
        'https://raw.githubusercontent.com/pytorch/examples/main/fast_neural_style/images/style-images/udnie.jpg'
    ],
    'Cezanne': [
        'https://commons.wikimedia.org/wiki/Special:FilePath/Paul_Cezanne_-_Mont_Sainte-Victoire_and_Ch%C3%A2teau_Noir_-_Google_Art_Project.jpg',
        'https://raw.githubusercontent.com/pytorch/examples/main/fast_neural_style/images/style-images/rain-princess.jpg'
    ],
}

STYLE_REFS, STYLE_HASHES = {}, {}

# Tracking which URL index resolves so we know if a fallback kicked in
STYLE_REF_URL_USED = {}

for _name, _urls in STYLE_REF_URLS.items():
    _slug = 'style_' + _name.replace(' ', '_').lower()
    _attempts = []

    # Iterating over primary and backup endpoints until the style image downloads cleanly
    for _idx, _u in enumerate(_urls):
        try:
            STYLE_REFS[_name], STYLE_HASHES[_name] = canonical_image(_u, _slug)
            STYLE_REF_URL_USED[_name] = _idx
            break
        except Exception as e:
            _attempts.append(f'{_u} -> {type(e).__name__}: {e}')

    if _name not in STYLE_REFS:
        raise RuntimeError(
            f'could not fetch style reference "{_name}": ' + '; '.join(_attempts)
        )

    # Logging whether we obtained the primary historical painting or hit a secondary target
    if STYLE_REF_URL_USED[_name] == 0:
        print(
            f'{_name:10s} {STYLE_REFS[_name].size} '
            f'sha256={STYLE_HASHES[_name]}  (primary URL, correct painting)'
        )
    else:
        print(f'{_name:10s} {STYLE_REFS[_name].size} sha256={STYLE_HASHES[_name]}')
        print(
            f'  WARNING: primary URL failed ({_attempts[0]}) -- '
            f'fallback URL #{STYLE_REF_URL_USED[_name]} was used instead. '
            f'This is a different image, so results should not be used.'
        )

# Enforcing strict provenance: refusing to run inference if any primary painting was replaced by a fallback
assert all(v == 0 for v in STYLE_REF_URL_USED.values()), (
    f'one or more style references used a fallback URL instead of the real painting: '
    f'{ {k: v for k, v in STYLE_REF_URL_USED.items() if v != 0} } -- '
    f'fix connectivity to the primary URLs and re-run this cell before generating anything downstream.'
)


def get_style_ref(name):
    # Retrieving cached style reference image by artist key
    if name not in STYLE_REFS:
        raise KeyError(f'no style reference for "{name}"')
    return STYLE_REFS[name]


# Validating that downloaded reference images match the exact sha256 hashes recorded in C1 verification
if C1_VERIFICATION is not None and 'style_hashes' in C1_VERIFICATION:
    for name in STYLE_REF_URLS:
        expected = C1_VERIFICATION['style_hashes'].get(name)

        if expected is not None:
            actual = STYLE_HASHES[name]
            assert actual[:len(expected)] == expected or actual == expected, (
                f'{name}: style reference hash does not match C1_VERIFICATION -- '
                f'refusing to proceed with an unverified reference'
            )

    print('style reference hashes cross-checked against C1_VERIFICATION: OK')
else:
    print(
        'C1_VERIFICATION not loaded (see Section 2) -- hashes recorded but not '
        'cross-checked against a live C1 run in this environment.'
    )

canonicalised style_van_gogh.png
Van Gogh   (512, 512) sha256=8d4ad662bacc42ba736574cc33a280944131c78fda5e8c549fd5f5f6492a3c1f  (primary URL, correct painting)
canonicalised style_cezanne.png
Cezanne    (512, 512) sha256=004c93bb56cdeace9462d7bb5b37c0617ab98810fd9570482de8b97976a282fd  (primary URL, correct painting)
C1_VERIFICATION not loaded (see Section 2) -- hashes recorded but not cross-checked against a live C1 run in this environment.


## 9. C1-Compatible Prompt Construction

Adopting the `ARTIST_STYLE_PHRASES` mapping and `region_style_prompt` logic directly from C1 rather than relying on earlier routing templates.Constructing the prompts using the exact C1 phrasing ensures textual conditioning remains consistent with the primary benchmark runs.

In [14]:
ARTIST_STYLE_PHRASES = {
    'Van Gogh': 'thick impasto brushwork, swirling expressive daylight strokes, saturated warm-cool contrast, Van Gogh style',
    'Cezanne': 'structured geometric brushwork, muted earthy greens and ochres, faceted planes of color, Cezanne Post-Impressionist style',
}
def region_style_prompt(region, style_name):
    desc = REGION_DESCRIPTORS.get(region, region)
    phrase = ARTIST_STYLE_PHRASES.get(style_name)
    return f'{desc}, {phrase}' if phrase else f'{desc} painted in {style_name} style'
for s in QUAL_SAMPLES:
    s['prompt1'] = region_style_prompt(s['region1'], s['style1_name'])
    s['prompt2'] = region_style_prompt(s['region2'], s['style2_name'])
    s['joint_prompt'] = f"{s['prompt1']}, {s['prompt2']}"
    print(f"{s['id']}:")
    print(f"  prompt1 = {s['prompt1']!r}")
    print(f"  prompt2 = {s['prompt2']!r}")

NEGATIVE_PROMPT = CONFIG['negative_prompt']
assert NEGATIVE_PROMPT == 'ugly, blurry, distorted, watermark, hard seam', \
    'negative prompt drifted from C1 -- must match exactly, not the old routing notebook\'s shorter string'
print('\nnegative prompt (C1 exact):', repr(NEGATIVE_PROMPT))

QUAL_01:
  prompt1 = 'sky, thick impasto brushwork, swirling expressive daylight strokes, saturated warm-cool contrast, Van Gogh style'
  prompt2 = 'mountain landscape, structured geometric brushwork, muted earthy greens and ochres, faceted planes of color, Cezanne Post-Impressionist style'
QUAL_02:
  prompt1 = 'sky, thick impasto brushwork, swirling expressive daylight strokes, saturated warm-cool contrast, Van Gogh style'
  prompt2 = 'mountain landscape, structured geometric brushwork, muted earthy greens and ochres, faceted planes of color, Cezanne Post-Impressionist style'
QUAL_03:
  prompt1 = 'sky, thick impasto brushwork, swirling expressive daylight strokes, saturated warm-cool contrast, Van Gogh style'
  prompt2 = 'mountain landscape, structured geometric brushwork, muted earthy greens and ochres, faceted planes of color, Cezanne Post-Impressionist style'

negative prompt (C1 exact): 'ugly, blurry, distorted, watermark, hard seam'


## 10. Proposed Method (F) Implementation

Implementing `generate_cross_attention` following the exact C1 configuration. This pipeline uses identical model checkpoints, inference steps, guidance scales, IP-Adapter weighting, and the `IPAdapterMaskProcessor`.Routing spatial style conditioning via `ip_adapter_masks` within `cross_attention_kwargs` allows the full multi-style composition to synthesize in a single diffusion pass.This setup mirrors the attention-steering mechanism analyzed in the frozen C2 experiments, where spatial attention localization was confirmed without needing to rerun those diagnostic passes here.

In [15]:
def load_models():
    require_gpu('model loading')
    if 'pipe' in MODELS:
        print('already loaded')
        return
    from diffusers import AutoPipelineForInpainting
    from diffusers.image_processor import IPAdapterMaskProcessor

    pipe = AutoPipelineForInpainting.from_pretrained(
        CONFIG['sd_repo'], torch_dtype=DTYPE, safety_checker=None,
        cache_dir=WEIGHTS_DIR).to(DEVICE)
    pipe.load_ip_adapter(CONFIG['ip_repo'], subfolder='models',
                         weight_name=[CONFIG['ip_weight']])
    pipe.set_ip_adapter_scale(CONFIG['ip_scale'])

    # Same activation check as C1: confirm the native cross-attention path is actually active.
    procs = {n: type(p).__name__ for n, p in pipe.unet.attn_processors.items()
             if n.endswith('attn2.processor')}
    assert procs and all(t == 'IPAdapterAttnProcessor2_0' for t in procs.values()), (
        f'expected IPAdapterAttnProcessor2_0 everywhere (the same processor class C2\'s '
        f'attention hooks measured), got {set(procs.values())}')
    assert pipe.unet.config.in_channels == 9, \
        f'expected the SD inpainting UNet (9 in_channels), got {pipe.unet.config.in_channels}'
    print(f'{len(procs)} cross-attention processors confirmed IPAdapterAttnProcessor2_0 '
          f'(same mechanism class C2 measured, per C2_ATTENTION_REFERENCE above)')
    print('scheduler:', type(pipe.scheduler).__name__)

    MODELS.update({
        'pipe': pipe,
        'mask_proc': IPAdapterMaskProcessor(),
    })
    print(f'proposed-F pipeline loaded, VRAM free {torch.cuda.mem_get_info()[0]/1e9:.1f} GB'
          if HAS_CUDA else 'proposed-F pipeline loaded (CPU)')


load_models()

model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

scheduler_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!
An error occurred while trying to fetch /mnt/vurm/homes/homes/rkr44/baselines_work/weights/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /mnt/vurm/homes/homes/rkr44/baselines_work/weights/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /mnt/vurm/homes/homes/rkr44/baselines_work/weights/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /mnt/vurm/homes/homes/rkr44/baselines_work/weights/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a7607

models/ip-adapter_sd15.bin:   0%|          | 0.00/44.6M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

models/image_encoder/model.safetensors:   0%|          | 0.00/2.53G [00:00<?, ?B/s]

16 cross-attention processors confirmed IPAdapterAttnProcessor2_0 (same mechanism class C2 measured, per C2_ATTENTION_REFERENCE above)
scheduler: DDIMScheduler
proposed-F pipeline loaded, VRAM free 4.4 GB


In [16]:
def make_ip_masks(m1, m2, h, w):
    # Preprocessing binary masks into tensor representations expected by IP-Adapter cross-attention
    proc = MODELS['mask_proc']
    t = proc.preprocess(
        [Image.fromarray(m1).convert('L'), Image.fromarray(m2).convert('L')],
        height=h, width=w
    )
    return [t.reshape(1, t.shape[0], t.shape[2], t.shape[3])]


def _gen(seed=None):
    # Initializing a dedicated PyTorch generator on the target device for reproducible sampling
    return torch.Generator(DEVICE).manual_seed(SEED if seed is None else seed)


def generate_cross_attention_F(content, m1, m2, s1, s2, prompt1, prompt2, seed=None):
    # Running Proposed Method F: single-pass diffusion with regional IP-Adapter cross-attention routing
    # Confining style features to target masks without requiring iterative inpainting
    pipe = MODELS['pipe']
    pipe.set_ip_adapter_scale([[CONFIG['ip_scale'], CONFIG['ip_scale']]])
    
    with torch.inference_mode():
        out = pipe(
            prompt=f'{prompt1}, {prompt2}',
            image=content,
            mask_image=Image.fromarray(np.maximum(m1, m2)).convert('L'),
            ip_adapter_image=[[s1, s2]],
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=CONFIG['steps'],
            guidance_scale=CONFIG['guidance'],
            strength=CONFIG['strength'],
            cross_attention_kwargs={'ip_adapter_masks': make_ip_masks(m1, m2, content.height, content.width)},
            generator=_gen(seed)
        ).images[0]
        
    # Resetting the IP-Adapter scales to default configuration after inference
    pipe.set_ip_adapter_scale([[CONFIG['ip_scale'], CONFIG['ip_scale']]])
    return out


print('proposed F defined: generate_cross_attention_F')

proposed F defined: generate_cross_attention_F


## 11. Baseline Implementations

We reviewed the baselines carried over from earlier exploratory notebooks to make sure each method slots cleanly into our shared data structures (`spec`, `content`, `m1`, `m2`, `s1`, `s2`) and respects the frozen configuration from Section 2.

Rather than relying on toy wrappers or mock outputs, each heavy baseline runs as a fully realized pipeline. When a method required practical engineering adjustments, fallback weights, or slight departures from the original paper, we logged the specifics directly in the summary table:

| Baseline | Status | Implementation Details |
|---|---|---|
| Sequential (naive two-pass) | VALID | Standard iterative inpainting across masks; running directly without modification. |
| IP-Adapter global | VALID | Conditioned using `CONFIG['ip_weight']` (`ip-adapter_sd15.bin`) to align with Method F, dropping earlier prototypes that pulled in `ip-adapter-plus_sd15.bin`. |
| IP-Adapter + rectangular masks | VALID | Running two consecutive 30-step passes (totalling 60 diffusion steps against Method F's 30), which we factor into our compute and latency comparisons. |
| NST (Gatys et al. 2016) | VALID | Optimization-based transfer via Caffe-VGG and L-BFGS, falling back to torchvision's VGG-19 if the legacy Caffe weights drop offline. |
| AdaIN (Huang & Belongie 2017) | VALID | Classic feed-forward autoencoder using pretrained checkpoints; omitted if the downloaded weights fail our integrity checksums. |
| InstructPix2Pix (Brooks et al. 2023) | VALID | Official checkpoint guided purely by natural-language instructions to establish an exemplar-free reference point. |
| MultiDiffusion (Bar-Tal et al. 2023) | VALID | Merges per-step spatial noise predictions across latent regions, hooking IP-Adapter style conditioning directly into the denoising loop. |
| ControlNet segmentation (Zhang et al. 2023) | VALID | Driven by our validated C1 masks instead of passing images through a separate off-the-shelf semantic segmenter. |
| SDXL text-only (Podell et al. 2023) | VALID | Benchmarking text-driven compositional quality without providing visual style references. |
| Regional attention-bias | VALID | Shifts cross-attention logits across spatial regions using fixed additive offsets rather than injecting full Prompt-to-Prompt attention maps. |
| RegionRoute | EXCLUDED | The public preprint lacks a verified, reproducible inference pipeline for our multi-mask setup, so we are leaving it out to avoid introducing untested reimplementations. |

We also checked whether a separate "hard compositing" baseline made sense, but a quick inspection showed it was just calling `baseline_sequential` under the hood. It produced byte-identical images across every single test sample, so treating it as an independent method would have been redundant.

To keep GPU memory stable on our machine, we split inference into two phases: the lighter baselines run inline per sample, while the heavier pipelines run in dedicated batches. Each larger model is loaded into VRAM once, processes all three qualitative images back-to-back, and gets flushed completely before the next architecture spins up.

In [17]:
def seed_all(seed):
    # Setting deterministic seeds across Python, NumPy, and PyTorch runtimes
    # Essential for NST L-BFGS optimizer, which operates outside Diffusers generators
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if HAS_CUDA:
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def sequential_inpaint(base_img, mask_arr, style_ref, prompt, seed):
    # Applying localized inpainting for a single region and style exemplar without cross attention routing
    pipe = MODELS['pipe']
    pipe.set_ip_adapter_scale(CONFIG['ip_scale'])
    with torch.inference_mode():
        out = pipe(
            prompt=prompt,
            image=base_img,
            mask_image=Image.fromarray(mask_arr).convert('L'),
            ip_adapter_image=style_ref,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=CONFIG['steps'],
            guidance_scale=CONFIG['guidance'],
            strength=CONFIG['strength'],
            generator=_gen(seed)
        ).images[0]
    # Restoring default dual stream adapter scale for subsequent pipeline calls
    pipe.set_ip_adapter_scale([[CONFIG['ip_scale'], CONFIG['ip_scale']]])
    return out

def baseline_sequential(spec, content, m1, m2, s1, s2, seed=None):
    # Running standard naive two pass sequential inpainting across both target regions
    out = content
    for mask, style, prompt in ((m1, s1, spec['prompt1']), (m2, s2, spec['prompt2'])):
        out = sequential_inpaint(out, mask, style, prompt, seed)
    return out

def baseline_ipa_global(spec, content, m1, m2, s1, s2, seed=None):
    # Running unrouted global IP Adapter conditioning across the merged mask
    # Ensuring CONFIG['ip_weight'] matches Method F so spatial routing remains the only independent variable
    pipe = MODELS['pipe']
    pipe.load_ip_adapter('h94/IP-Adapter', subfolder='models', weight_name=[CONFIG['ip_weight']])
    pipe.set_ip_adapter_scale(CONFIG['ip_scale'])
    try:
        with torch.inference_mode():
            out = pipe(
                prompt=spec['joint_prompt'],
                image=content,
                mask_image=Image.fromarray(np.maximum(m1, m2)).convert('L'),
                ip_adapter_image=[[s1, s2]],
                negative_prompt=NEGATIVE_PROMPT,
                num_inference_steps=CONFIG['steps'],
                guidance_scale=CONFIG['guidance'],
                strength=CONFIG['strength'],
                generator=_gen(seed)
            ).images[0]
    finally:
        # Resetting the per region adapter scale structure expected by Method F
        pipe.set_ip_adapter_scale([[CONFIG['ip_scale'], CONFIG['ip_scale']]])
    return out

def baseline_ipa_rect(spec, content, m1, m2, s1, s2, seed=None):
    # Running two pass inpainting with naive geometric rectangular splits instead of learned SAM masks
    # Accounting for the doubled compute budget (60 total diffusion steps vs 30 in Method F)
    H = m1.shape[0]
    cut = int(H * 0.5)
    ra = np.zeros_like(m1); ra[:cut] = 255
    rb = np.zeros_like(m2); rb[cut:] = 255
    pipe = MODELS['pipe']
    pipe.set_ip_adapter_scale(CONFIG['ip_scale'])
    try:
        cur = content
        for m, ref, pr in ((ra, s1, spec['prompt1']), (rb, s2, spec['prompt2'])):
            with torch.inference_mode():
                cur = pipe(
                    prompt=pr,
                    image=cur,
                    mask_image=Image.fromarray(m).convert('L'),
                    ip_adapter_image=ref,
                    negative_prompt=NEGATIVE_PROMPT,
                    num_inference_steps=CONFIG['steps'],
                    guidance_scale=CONFIG['guidance'],
                    strength=CONFIG['strength'],
                    generator=_gen(seed)
                ).images[0]
        return cur
    finally:
        # Restoring dual region adapter scaling for subsequent pipeline executions
        pipe.set_ip_adapter_scale([[CONFIG['ip_scale'], CONFIG['ip_scale']]])

VALID_BASELINES = {
    'Sequential (naive two-pass)': baseline_sequential,
    'IP-Adapter global (no routing)': baseline_ipa_global,
    'IP-Adapter + rect masks': baseline_ipa_rect,
}

# Heavy baselines requiring dedicated architectures or separate execution passes.
# Handled in dedicated stages to prevent redundant VRAM allocations during testing.
HEAVY_MODELS = {}

# NST (Gatys et al., CVPR 2016)
# Classical optimization using Caffe converted VGG 19 features with L-BFGS
# Retaining average pooling layers as specified in the original paper formulation
CAFFE_VGG_URL = 'https://web.eecs.umich.edu/~justincj/models/vgg19-d01eb7cb.pth'

# Tracking VGG checkpoint lineage across garbage collection cycles
NST_VGG_LOG = {}

def _load_nst_vgg():
    # Attempting to load Justin Johnson's Caffe converted VGG 19 weights for strict paper fidelity
    # Falling back to torchvision ImageNet weights if the remote host is unreachable
    if 'nst_vgg_ok' in HEAVY_MODELS:
        return HEAVY_MODELS['nst_vgg_ok']
    import torchvision.models as tvm
    caffe_path = f'{WEIGHTS_DIR}/vgg19_caffe.pth'
    ok, reason = False, None
    if not os.path.exists(caffe_path):
        try:
            import urllib.request
            req = urllib.request.Request(CAFFE_VGG_URL, headers={'User-Agent': 'Mozilla/5.0'})
            tmp = caffe_path + '.part'
            with urllib.request.urlopen(req, timeout=300) as r, open(tmp, 'wb') as f:
                f.write(r.read())
            os.replace(tmp, caffe_path)
        except Exception as e:
            reason = f'download failed ({type(e).__name__}: {e})'
    if reason is None:
        try:
            tvm.vgg19().load_state_dict(torch.load(caffe_path, map_location='cpu', weights_only=False))
            ok = True
        except Exception as e:
            reason = f'weights on disk do not load into torchvision vgg19 ({type(e).__name__}: {e})'
    if ok:
        print(f'NST VGG-19 branch: CAFFE-converted weights (Johnson), '
              f'sha256={sha256_file(caffe_path)[:12]} (faithful to Gatys et al.)')
    else:
        print(f'NST VGG-19 branch: FALLBACK torchvision ImageNet VGG-19 ({reason}). '
              f'Disclose this substitution in the write-up.')
    NST_VGG_LOG.update({
        'variant': 'caffe' if ok else 'torchvision_imagenet_fallback',
        'fallback_reason': reason,
        'weights_sha256': sha256_file(caffe_path) if ok else None,
        'logged_at': datetime.now().isoformat()
    })
    HEAVY_MODELS['nst_vgg_ok'] = ok
    return ok

def baseline_nst(spec, content, m1, m2, s1, s2, seed=None,
                 max_iters=1000, style_w=1e6, content_w=1.0, tol=1e-4, patience=5):
    # Executing full iterative L-BFGS style transfer per region using intermediate VGG representations
    import torchvision.models as tvm
    import torchvision.transforms as T
    seed_all(seed if seed is not None else CONFIG['seed'])
    caffe_ok = _load_nst_vgg()
    # Instantiating the feature backbone based on verified weight availability
    if caffe_ok:
        vgg_full = tvm.vgg19()
        vgg_full.load_state_dict(torch.load(f'{WEIGHTS_DIR}/vgg19_caffe.pth',
                                            map_location='cpu', weights_only=False))
        vgg = vgg_full.features
    else:
        vgg = tvm.vgg19(weights=tvm.VGG19_Weights.DEFAULT).features
    # Replacing standard MaxPool operations with AvgPool as recommended by Gatys et al.
    for idx, m in enumerate(vgg):
        if isinstance(m, torch.nn.MaxPool2d):
            vgg[idx] = torch.nn.AvgPool2d(2, 2)
    vgg = vgg.to(DEVICE).eval()
    for p in vgg.parameters():
        p.requires_grad_(False)
    # Configuring normalization transforms and color spaces matching the selected VGG branch
    if caffe_ok:
        CAFFE_MEAN = torch.tensor([103.939, 116.779, 123.680], device=DEVICE).view(1, 3, 1, 1)
        def to_net(img):
            arr = np.array(img.convert('RGB')).astype(np.float32)[:, :, ::-1].copy()
            t = torch.from_numpy(arr).permute(2, 0, 1).unsqueeze(0).to(DEVICE)
            return t - CAFFE_MEAN
        def from_net(t):
            arr = (t.squeeze(0) + CAFFE_MEAN).clamp(0, 255)
            arr = arr.permute(1, 2, 0).cpu().numpy().astype(np.uint8)
            return Image.fromarray(arr[:, :, ::-1].copy())
        c_lo, c_hi = -float(CAFFE_MEAN.max()), 255 - float(CAFFE_MEAN.min())
    else:
        mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)
        tf = T.ToTensor()
        def to_net(img):
            return (tf(img.convert('RGB')).unsqueeze(0).to(DEVICE) - mean) / std
        def from_net(t):
            arr = (t.detach() * std + mean).clamp(0, 1)
            return Image.fromarray((arr.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8))
        c_lo, c_hi = -2.5, 2.5
    # Selecting feature layers for style correlation and content structure
    SL = {'0', '5', '10', '19', '28'}
    CL = '21'
    def feats(x):
        o, h = {}, x
        for n, l in vgg._modules.items():
            h = l(h)
            if n in SL or n == CL:
                o[n] = h
        return o
    def gram(f):
        # Computing normalized Gram matrix over spatial feature dimensions
        _, c, h, w = f.shape
        m = f.view(c, h * w)
        return m @ m.t() / (c * h * w)
    def render(cimg, simg):
        # Optimizing an input canvas via L-BFGS to jointly balance content loss and style Gram statistics
        c = to_net(cimg); st = to_net(simg)
        with torch.no_grad():
            cf = feats(c); sf = feats(st)
            sg = {k: gram(sf[k]) for k in SL}
        tgt = c.clone().requires_grad_(True)
        opt = torch.optim.LBFGS([tgt], max_iter=20, lr=1.0)
        hist, stalled = [], 0
        while len(hist) * 20 < max_iters:
            def closure():
                opt.zero_grad()
                f = feats(tgt)
                loss = (content_w * ((f[CL] - cf[CL]) ** 2).mean()
                        + style_w * sum(((gram(f[k]) - sg[k]) ** 2).mean() for k in SL))
                loss.backward()
                return loss
            hist.append(float(opt.step(closure)))
            with torch.no_grad():
                tgt.clamp_(c_lo, c_hi)
            # Checking convergence criteria to terminate optimization early if progress plateaus
            if len(hist) > 1 and abs(hist[-2] - hist[-1]) / max(hist[-2], 1e-8) < tol:
                stalled += 1
                if stalled >= patience:
                    break
            else:
                stalled = 0
        return from_net(tgt.detach())
    # Rendering separate stylized variants and combining them according to the regional segmentation masks
    oa = render(content, s1)
    ob = render(content, s2)
    out = np.array(content).copy()
    out[m1 > 127] = np.array(oa)[m1 > 127]
    out[m2 > 127] = np.array(ob)[m2 > 127]
    del vgg
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
    return Image.fromarray(out)

# AdaIN (Huang & Belongie, ICCV 2017)
# Feed forward style transfer via adaptive instance normalization in feature space
ADAIN_URLS = {
    'decoder': 'https://huggingface.co/spaces/tidalove/adain/resolve/main/decoder.pth',
    'vgg': 'https://huggingface.co/spaces/tidalove/adain/resolve/main/vgg_normalized.pth',
}

def _adain_transform(cf, sf, eps=1e-5):
    # Aligning channel wise mean and variance of content representations to match the style exemplar
    cm = cf.mean([2, 3], keepdim=True); cs = cf.std([2, 3], keepdim=True) + eps
    sm = sf.mean([2, 3], keepdim=True); ss = sf.std([2, 3], keepdim=True) + eps
    return ss * (cf - cm) / cs + sm

def _build_adain_encoder(state, device):
    # Constructing truncated VGG encoder targeting relu4_1 activations
    def conv(w, b, padding=0):
        c = torch.nn.Conv2d(w.shape[1], w.shape[0], w.shape[2], padding=padding, bias=True)
        with torch.no_grad():
            c.weight.copy_(w); c.bias.copy_(b)
        return c
    s = state
    layers = [
        conv(s['0.weight'], s['0.bias'], padding=0),
        conv(s['1.weight'], s['1.bias'], padding=1), torch.nn.ReLU(inplace=True),
        conv(s['3.weight'], s['3.bias'], padding=1), torch.nn.ReLU(inplace=True),
        torch.nn.MaxPool2d(2, 2),
        conv(s['6.weight'], s['6.bias'], padding=1), torch.nn.ReLU(inplace=True),
        conv(s['8.weight'], s['8.bias'], padding=1), torch.nn.ReLU(inplace=True),
        torch.nn.MaxPool2d(2, 2),
        conv(s['11.weight'], s['11.bias'], padding=1), torch.nn.ReLU(inplace=True),
        conv(s['13.weight'], s['13.bias'], padding=1), torch.nn.ReLU(inplace=True),
        conv(s['15.weight'], s['15.bias'], padding=1), torch.nn.ReLU(inplace=True),
        conv(s['17.weight'], s['17.bias'], padding=1), torch.nn.ReLU(inplace=True),
        torch.nn.MaxPool2d(2, 2),
        conv(s['20.weight'], s['20.bias'], padding=1), torch.nn.ReLU(inplace=True),
    ]
    return torch.nn.Sequential(*layers).to(device).eval()

def _build_adain_decoder(state, device):
    # Constructing feed forward decoder to invert transformed feature maps back to RGB
    def rconv(w, b):
        pad = w.shape[2] // 2
        c = torch.nn.Sequential(
            torch.nn.ReflectionPad2d(pad),
            torch.nn.Conv2d(w.shape[1], w.shape[0], w.shape[2], bias=True)
        )
        with torch.no_grad():
            c[1].weight.copy_(w); c[1].bias.copy_(b)
        return c
    s = state
    up = torch.nn.Upsample(scale_factor=2, mode='nearest')
    layers = [
        rconv(s['0.weight'], s['0.bias']), torch.nn.ReLU(inplace=True), up,
        rconv(s['3.weight'], s['3.bias']), torch.nn.ReLU(inplace=True),
        rconv(s['5.weight'], s['5.bias']), torch.nn.ReLU(inplace=True),
        rconv(s['7.weight'], s['7.bias']), torch.nn.ReLU(inplace=True),
        rconv(s['9.weight'], s['9.bias']), torch.nn.ReLU(inplace=True), up,
        rconv(s['12.weight'], s['12.bias']), torch.nn.ReLU(inplace=True),
        rconv(s['14.weight'], s['14.bias']), torch.nn.ReLU(inplace=True), up,
        rconv(s['17.weight'], s['17.bias']), torch.nn.ReLU(inplace=True),
        rconv(s['19.weight'], s['19.bias']),
    ]
    return torch.nn.Sequential(*layers).to(device).eval()

def _load_adain():
    # Fetching pretrained encoder and decoder weights if not already present on disk
    if 'adain_ok' in HEAVY_MODELS:
        return HEAVY_MODELS['adain_ok']
    import urllib.request
    dec_path = f'{WEIGHTS_DIR}/adain_decoder.pth'
    vgg_path = f'{WEIGHTS_DIR}/adain_vgg_normalized.pth'
    try:
        for dest, url in ((dec_path, ADAIN_URLS['decoder']), (vgg_path, ADAIN_URLS['vgg'])):
            if os.path.exists(dest):
                continue
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=300) as r, open(dest, 'wb') as f:
                f.write(r.read())
        enc_state = torch.load(vgg_path, map_location='cpu', weights_only=False)
        dec_state = torch.load(dec_path, map_location='cpu', weights_only=False)
        HEAVY_MODELS['adain_enc'] = _build_adain_encoder(enc_state, DEVICE)
        HEAVY_MODELS['adain_dec'] = _build_adain_decoder(dec_state, DEVICE)
        HEAVY_MODELS['adain_ok'] = True
    except Exception as e:
        print(f'AdaIN: weight download/build failed ({e}) (AdaIN excluded, not faked)')
        HEAVY_MODELS['adain_ok'] = False
    return HEAVY_MODELS['adain_ok']

def baseline_adain(spec, content, m1, m2, s1, s2, seed=None, per_region=True, alpha=1.0):
    # Executing feed forward style transfer via AdaIN feature normalization across spatial segments
    import torchvision.transforms as T
    if not _load_adain():
        raise RuntimeError('AdaIN weights unavailable (see printed message; not faked)')
    ae, ad = HEAVY_MODELS['adain_enc'], HEAVY_MODELS['adain_dec']
    tf = T.Compose([T.Resize(CONFIG['image_size']), T.ToTensor()])
    def go(cimg, simg):
        ct = tf(cimg.convert('RGB')).unsqueeze(0).to(DEVICE)
        stt = tf(simg.convert('RGB')).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            cf = ae(ct)
            t = _adain_transform(cf, ae(stt))
            t = alpha * t + (1 - alpha) * cf
            out = ad(t).clamp(0, 1)
        arr = (out.squeeze(0).permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
        return Image.fromarray(arr)
    # Performing forward passes per style reference and compositing according to regional masks
    out_a, out_b = go(content, s1), go(content, s2)
    canvas = np.array(content).copy()
    canvas[m1 > 127] = np.array(out_a)[m1 > 127]
    canvas[m2 > 127] = np.array(out_b)[m2 > 127]
    return Image.fromarray(canvas)

# InstructPix2Pix (Brooks et al., 2023)
def baseline_ip2p(spec, content, m1, m2, s1, s2, seed=None, steps=100):
    # Running prompt guided image to image editing using official InstructPix2Pix weights
    # Note: operates strictly from natural language instructions without exemplar style images
    from diffusers import StableDiffusionInstructPix2PixPipeline, EulerAncestralDiscreteScheduler
    if 'ip2p' not in HEAVY_MODELS:
        HEAVY_MODELS['ip2p'] = StableDiffusionInstructPix2PixPipeline.from_pretrained(
            'timbrooks/instruct-pix2pix', torch_dtype=DTYPE, safety_checker=None,
            cache_dir=WEIGHTS_DIR
        ).to(DEVICE)
        HEAVY_MODELS['ip2p'].scheduler = EulerAncestralDiscreteScheduler.from_config(
            HEAVY_MODELS['ip2p'].scheduler.config
        )
    pipe = HEAVY_MODELS['ip2p']
    assert type(pipe.scheduler).__name__ == 'EulerAncestralDiscreteScheduler'
    # Constructing joint instruction targeting both regions simultaneously
    instruction = (f"Paint the {spec['region1']} in the style of {spec['style1_name']} "
                   f"and paint the {spec['region2']} in the style of {spec['style2_name']}")
    return pipe(
        instruction,
        image=content,
        num_inference_steps=steps,
        image_guidance_scale=1.5,
        guidance_scale=7.5,
        negative_prompt=NEGATIVE_PROMPT,
        generator=_gen(seed)
    ).images[0]

print('baseline definitions loaded:', list(VALID_BASELINES),
      '+ heavy: NST, AdaIN, InstructPix2Pix (run separately in Section 13)')

baseline definitions loaded: ['Sequential (naive two-pass)', 'IP-Adapter global (no routing)', 'IP-Adapter + rect masks'] + heavy: NST, AdaIN, InstructPix2Pix (run separately in Section 13)


In [18]:
# Loading heavy baselines that require architectural hooks or dedicated pipelines
# MultiDiffusion and regional attention bias reuse pipe_native directly
# ControlNet and SDXL offload pipe_native to CPU temporarily to stay within GPU memory limits

import torch.nn.functional as F

def run_multidiffusion(spec, content, m1, m2, s1, s2, seed=None):
    # Fusing regional noise estimates at each denoising step across a shared latent trajectory
    # Uncovered areas receive equal 0.5 weights to avoid zeroed predictions that confuse the scheduler
    pipe = MODELS['pipe']
    unet, vae, sched = pipe.unet, pipe.vae, pipe.scheduler
    vs = 2 ** (len(vae.config.block_out_channels) - 1)
    dtype = unet.dtype

    ea, na = pipe.encode_prompt(spec['prompt1'], DEVICE, 1, True, negative_prompt=NEGATIVE_PROMPT)
    eb, nb = pipe.encode_prompt(spec['prompt2'], DEVICE, 1, True, negative_prompt=NEGATIVE_PROMPT)
    ta, tb = torch.cat([na, ea]), torch.cat([nb, eb])

    pipe.set_ip_adapter_scale(CONFIG['ip_scale'])
    try:
        ipa = pipe.prepare_ip_adapter_image_embeds(s1, None, DEVICE, 1, True)
        ipb = pipe.prepare_ip_adapter_image_embeds(s2, None, DEVICE, 1, True)

        def lat(m):
            t = torch.from_numpy((m > 127).astype(np.float32))[None, None].to(DEVICE, dtype)
            return F.interpolate(t, scale_factor=1 / vs, mode='nearest')

        # Downsampling regional masks to match latent dimensions and computing normalized blend weights
        la, lb = lat(m1), lat(m2)
        covered = (la + lb) > 0
        denom = (la + lb).clamp(min=1e-6)
        w_a = torch.where(covered, la / denom, torch.full_like(la, 0.5))
        w_b = torch.where(covered, lb / denom, torch.full_like(lb, 0.5))
        assert torch.allclose(w_a + w_b, torch.ones_like(w_a)), 'weights must sum to 1'

        union = np.maximum(m1, m2)
        cm = torch.cat([lat(union)] * 2)
        mk = np.array(content).astype(np.float32) / 127.5 - 1.0
        mk[union > 127] = 0.0
        mt = torch.from_numpy(mk).permute(2, 0, 1).unsqueeze(0).to(DEVICE, dtype)
        with torch.no_grad():
            ml = vae.encode(mt).latent_dist.sample() * vae.config.scaling_factor
        ml = torch.cat([ml] * 2)

        H, W = content.size[1], content.size[0]
        g = _gen(seed)
        z = torch.randn((1, vae.config.latent_channels, H // vs, W // vs),
                        generator=g, device=DEVICE, dtype=dtype)
        sched.set_timesteps(CONFIG['steps'], device=DEVICE)
        z = z * sched.init_noise_sigma

        # Stepping through diffusion timesteps while blending regional noise predictions in latent space
        with torch.no_grad():
            for t in sched.timesteps:
                li = sched.scale_model_input(torch.cat([z] * 2), t)
                ui = torch.cat([li, cm, ml], dim=1)
                na_ = unet(ui, t, encoder_hidden_states=ta, added_cond_kwargs={'image_embeds': ipa}).sample
                nb_ = unet(ui, t, encoder_hidden_states=tb, added_cond_kwargs={'image_embeds': ipb}).sample
                nu_a, nc_a = na_.chunk(2)
                nu_b, nc_b = nb_.chunk(2)
                pa = nu_a + CONFIG['guidance'] * (nc_a - nu_a)
                pb = nu_b + CONFIG['guidance'] * (nc_b - nu_b)
                z = sched.step(pa * w_a + pb * w_b, t, z).prev_sample
            img = vae.decode(z / vae.config.scaling_factor).sample
        img = (img / 2 + 0.5).clamp(0, 1).squeeze(0).permute(1, 2, 0).float().cpu().numpy()
        return Image.fromarray((img * 255).round().astype(np.uint8))
    finally:
        pipe.set_ip_adapter_scale([[CONFIG['ip_scale'], CONFIG['ip_scale']]])

# ControlNet segmentation (Zhang et al., ICCV 2023)
def _ade_palette():
    return [
        [0, 0, 0], [120, 120, 120], [180, 120, 120], [6, 230, 230], [80, 50, 50],
        [4, 200, 3], [120, 120, 80], [140, 140, 140], [204, 5, 255], [230, 230, 230],
        [4, 250, 7], [224, 5, 255], [235, 255, 7], [150, 5, 61], [120, 120, 70],
        [8, 255, 51], [255, 6, 82], [143, 255, 140], [204, 255, 4], [255, 51, 7],
    ]

# Mapping sky and mountain labels to ADE20K color indices with zero index offset applied
ADE_REGION1_IDX = 3    # ADE20K sky: [6, 230, 230]
ADE_REGION2_IDX = 17   # ADE20K mountain: [143, 255, 140]

def run_controlnet(spec, content, m1, m2, s1, s2, seed=None):
    # Generating an ADE20K color-coded conditioning map from the verified regional masks
    from diffusers import ControlNetModel, StableDiffusionControlNetInpaintPipeline
    if 'controlnet_pipe' not in HEAVY_MODELS:
        if 'pipe' in MODELS:
            MODELS['pipe'].to('cpu')
            gc.collect()
            if HAS_CUDA:
                torch.cuda.empty_cache()
        controlnet = ControlNetModel.from_pretrained(
            'lllyasviel/control_v11p_sd15_seg', torch_dtype=DTYPE, cache_dir=WEIGHTS_DIR)
        HEAVY_MODELS['controlnet_pipe'] = StableDiffusionControlNetInpaintPipeline.from_pretrained(
            CONFIG['sd_repo'], controlnet=controlnet, torch_dtype=DTYPE,
            safety_checker=None, cache_dir=WEIGHTS_DIR).to(DEVICE)

    pipe_cn = HEAVY_MODELS['controlnet_pipe']
    palette = _ade_palette()
    assert (palette[ADE_REGION1_IDX] == [6, 230, 230]
            and palette[ADE_REGION2_IDX] == [143, 255, 140]), (
        'ADE palette indices no longer point at sky/mountain: seg map would mislabel regions')
    seg_map = np.zeros((*m1.shape, 3), dtype=np.uint8)
    seg_map[m1 > 127] = palette[ADE_REGION1_IDX]
    seg_map[m2 > 127] = palette[ADE_REGION2_IDX]
    seg_control = Image.fromarray(seg_map)

    return pipe_cn(
        prompt=spec['joint_prompt'], image=content,
        mask_image=Image.fromarray(np.maximum(m1, m2)).convert('L'),
        control_image=seg_control, negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=CONFIG['steps'], guidance_scale=CONFIG['guidance'],
        generator=_gen(seed)).images[0]

def _release_controlnet():
    # Evicting ControlNet from VRAM and returning pipe_native back to GPU memory
    if 'controlnet_pipe' in HEAVY_MODELS:
        del HEAVY_MODELS['controlnet_pipe']
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
    if 'pipe' in MODELS:
        MODELS['pipe'].to(DEVICE)

# SDXL text-only (Podell et al., 2023)
def run_sdxl(spec, content, m1, m2, s1, s2, seed=None):
    # Running native 1024x1024 prompt guided inpainting without image style references
    from diffusers import AutoPipelineForInpainting
    if 'sdxl_pipe' not in HEAVY_MODELS:
        if 'pipe' in MODELS:
            MODELS['pipe'].to('cpu')
            gc.collect()
            if HAS_CUDA:
                torch.cuda.empty_cache()
        HEAVY_MODELS['sdxl_pipe'] = AutoPipelineForInpainting.from_pretrained(
            'diffusers/stable-diffusion-xl-1.0-inpainting-0.1',
            torch_dtype=DTYPE, cache_dir=WEIGHTS_DIR).to(DEVICE)

    pipe_sdxl = HEAVY_MODELS['sdxl_pipe']
    img = pipe_sdxl(
        prompt=spec['joint_prompt'], image=content,
        mask_image=Image.fromarray(np.maximum(m1, m2)).convert('L'),
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=CONFIG['steps'], guidance_scale=CONFIG['guidance'],
        generator=_gen(seed)).images[0]
    return img.resize(content.size, Image.LANCZOS)

def _release_sdxl():
    # Tearing down SDXL pipeline to free VRAM for downstream modules
    if 'sdxl_pipe' in HEAVY_MODELS:
        del HEAVY_MODELS['sdxl_pipe']
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()
    if 'pipe' in MODELS:
        MODELS['pipe'].to(DEVICE)

# Regional attention bias
class RegionalPromptAttnProcessor:
    # Adding constant positive logit offsets to specific token cross attention scores across spatial masks
    def __init__(self, bias_map):
        self.bias_map = bias_map

    def __call__(self, attn, hidden_states, encoder_hidden_states=None,
                 attention_mask=None, temb=None, *args, **kwargs):
        residual, ndim = hidden_states, hidden_states.ndim
        if ndim == 4:
            bsz, ch, h, w = hidden_states.shape
            hidden_states = hidden_states.view(bsz, ch, h * w).transpose(1, 2)
        else:
            bsz, seq, _ = hidden_states.shape
            side = int(round(seq ** 0.5)); h = w = side

        is_cross = encoder_hidden_states is not None
        if isinstance(encoder_hidden_states, (tuple, list)):
            encoder_hidden_states = encoder_hidden_states[0]
        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states

        q = attn.head_to_batch_dim(attn.to_q(hidden_states))
        k = attn.head_to_batch_dim(attn.to_k(encoder_hidden_states))
        v = attn.head_to_batch_dim(attn.to_v(encoder_hidden_states))
        sc = torch.baddbmm(torch.empty(q.shape[0], q.shape[1], k.shape[1], dtype=q.dtype, device=q.device),
                           q, k.transpose(-1, -2), beta=0, alpha=attn.scale)

        # Applying spatial logit shifts only to conditional attention heads
        if is_cross and encoder_hidden_states.shape[1] == self.bias_map.shape[0]:
            b = F.interpolate(self.bias_map.permute(1, 0, 2, 3), size=(h, w), mode='nearest')
            b = b.reshape(1, self.bias_map.shape[0], h * w).permute(0, 2, 1)
            n = sc.shape[0]
            full = torch.zeros(n, b.shape[1], b.shape[2], dtype=sc.dtype, device=sc.device)
            full[n // 2:] = b.to(sc.dtype)
            sc = sc + full

        out = attn.batch_to_head_dim(torch.bmm(sc.softmax(dim=-1), v))
        out = attn.to_out[1](attn.to_out[0](out))
        if ndim == 4:
            out = out.transpose(-1, -2).reshape(bsz, ch, h, w)
        if attn.residual_connection:
            out = out + residual
        return out / attn.rescale_output_factor

def _find_token_span(tokenizer, prompt, phrase):
    # Locating exact token indices for target artist names within the padded text embedding sequence
    ids = tokenizer(prompt, padding='max_length', max_length=tokenizer.model_max_length,
                    truncation=True, return_tensors='pt').input_ids[0].tolist()
    if getattr(tokenizer, 'is_fast', False):
        char_start = prompt.find(phrase)
        if char_start != -1:
            enc = tokenizer(prompt, padding='max_length', max_length=tokenizer.model_max_length,
                            truncation=True, return_offsets_mapping=True)
            char_end = char_start + len(phrase)
            span = [i for i, (s, e) in enumerate(enc['offset_mapping'])
                    if s < char_end and e > char_start and e > s]
            if span:
                return span
    for variant in (phrase, ' ' + phrase):
        pids = tokenizer(variant, add_special_tokens=False).input_ids
        for start in range(len(ids) - len(pids) + 1):
            if ids[start:start + len(pids)] == pids:
                return list(range(start, start + len(pids)))
    return []

def run_regional_attention_bias(spec, content, m1, m2, s1, s2, seed=None):
    # Injecting custom cross attention processors to bias text token influence toward respective regions
    pipe = MODELS['pipe']
    prompt = spec['joint_prompt']
    tokenizer = pipe.tokenizer
    r1_toks = _find_token_span(tokenizer, prompt, spec['style1_name'])
    r2_toks = _find_token_span(tokenizer, prompt, spec['style2_name'])
    if not r1_toks or not r2_toks:
        print(f"  WARNING {spec['id']}: empty token span for one style name: bias is a no-op for that region")

    la = F.interpolate(torch.from_numpy((m1 > 127).astype(np.float32))[None, None].to(DEVICE),
                       size=(64, 64), mode='nearest')
    lb = F.interpolate(torch.from_numpy((m2 > 127).astype(np.float32))[None, None].to(DEVICE),
                       size=(64, 64), mode='nearest')
    bias_map = torch.zeros(77, 1, 64, 64, device=DEVICE, dtype=pipe.unet.dtype)
    for tok in r1_toks:
        if tok < 77:
            bias_map[tok] = la.to(bias_map.dtype) * 6.0
    for tok in r2_toks:
        if tok < 77:
            bias_map[tok] = lb.to(bias_map.dtype) * 6.0

    orig_procs = pipe.unet.attn_processors.copy()
    rp = RegionalPromptAttnProcessor(bias_map)
    new_procs = {n: (rp if n.endswith('attn2.processor') else p) for n, p in orig_procs.items()}
    try:
        pipe.unet.set_attn_processor(new_procs)
        pipe.set_ip_adapter_scale(0.0)
        return pipe(
            prompt=prompt, image=content,
            mask_image=Image.fromarray(np.maximum(m1, m2)).convert('L'),
            ip_adapter_image=s1,
            negative_prompt=NEGATIVE_PROMPT,
            num_inference_steps=CONFIG['steps'], guidance_scale=CONFIG['guidance'],
            generator=_gen(seed)).images[0]
    finally:
        # Reverting back to original cross attention processors and resetting adapter scales
        pipe.unet.set_attn_processor(orig_procs)
        pipe.set_ip_adapter_scale([[CONFIG['ip_scale'], CONFIG['ip_scale']]])

HEAVY_BASELINES = {
    'NST': {'fn': baseline_nst, 'teardown': None},
    'AdaIN': {'fn': baseline_adain, 'teardown': None},
    'InstructPix2Pix': {'fn': baseline_ip2p, 'teardown': None},
    'MultiDiffusion': {'fn': run_multidiffusion, 'teardown': None},
    'ControlNet segmentation': {'fn': run_controlnet, 'teardown': _release_controlnet},
    'SDXL text-only': {'fn': run_sdxl, 'teardown': _release_sdxl},
    'Regional attention-bias (not Prompt-to-Prompt)': {'fn': run_regional_attention_bias, 'teardown': None},
}
RUN_HEAVY_BASELINES = {name: True for name in HEAVY_BASELINES}

print('heavy baseline definitions loaded:', list(HEAVY_BASELINES))
print('RUN_HEAVY_BASELINES (toggle any to False to skip):', RUN_HEAVY_BASELINES)

heavy baseline definitions loaded: ['NST', 'AdaIN', 'InstructPix2Pix', 'MultiDiffusion', 'ControlNet segmentation', 'SDXL text-only', 'Regional attention-bias (not Prompt-to-Prompt)']
RUN_HEAVY_BASELINES (toggle any to False to skip): {'NST': True, 'AdaIN': True, 'InstructPix2Pix': True, 'MultiDiffusion': True, 'ControlNet segmentation': True, 'SDXL text-only': True, 'Regional attention-bias (not Prompt-to-Prompt)': True}


## 12. Export Bridge Files for the Standalone BrushEdit Notebook

Running BrushEdit as an independent pipeline (`BrushEdit_for_dissertation.ipynb`) on Colab completely detached from the Part 1 and Part 2 execution sessions. This section packages and exports the exact required inputs (content images, validated C1 masks, and formatted prompts) so the downstream results remain directly comparable without needing shared runtime state.Placing the export step here rather than in Part 2 allows BrushEdit to kick off as soon as Part 1 wraps up. Because it only depends on our verified masks and prompt structures, it can execute entirely in parallel while Part 2 handles the heavier baselines, leaving final outputs to be consolidated from their respective archives later.

In [28]:
# Exporting content images, C1 masks, and text prompts for the standalone BrushEdit pipeline
BRUSHEDIT_BRIDGE_DIR = f'{WORK}/brushedit_bridge'
os.makedirs(BRUSHEDIT_BRIDGE_DIR, exist_ok=True)

def _npy_matches_png(sid, m1, m2):
    # Verifying binary consistency between exported NumPy arrays and saved PNG reference masks
    rec = QUAL_MASK_META[sid]
    ok = all(np.array_equal(np.array(Image.open(rec[k]).convert('L')) > 127, m > 127)
             for k, m in (('p1', m1), ('p2', m2)))
    if not ok:
        print(f'  WARNING {sid}: exported .npy mask differs from the PNG whose hash is recorded')
    return bool(ok)

def export_brushedit_bridge():
    # Serializing inputs, hyperparameter metadata, and sha256 checksums to manifest.json
    manifest = {}
    for spec in QUAL_SAMPLES:
        sid = spec['id']
        content_path = f'{BRUSHEDIT_BRIDGE_DIR}/{sid}_content.png'
        mask1_path = f'{BRUSHEDIT_BRIDGE_DIR}/{sid}_mask_region1.npy'
        mask2_path = f'{BRUSHEDIT_BRIDGE_DIR}/{sid}_mask_region2.npy'
        
        QUAL_CONTENT[sid].save(content_path, 'PNG')
        m1, m2 = QUAL_MASKS[sid]
        np.save(mask1_path, m1)
        np.save(mask2_path, m2)
        
        # Logging experiment settings and file hashes for downstream verification
        manifest[sid] = {
            'content': content_path,
            'mask_region1': mask1_path,
            'mask_region2': mask2_path,
            'prompt1': spec['prompt1'],
            'prompt2': spec['prompt2'],
            'negative': NEGATIVE_PROMPT,
            'seed': CONFIG['seed'],
            'steps': CONFIG['steps'],
            'guidance': CONFIG['guidance'],
            'content_sha256': sha256_file(content_path),
            'mask1_sha256': QUAL_MASK_META[sid]['h1'],
            'mask2_sha256': QUAL_MASK_META[sid]['h2'],
            'mask1_npy_sha256': sha256_file(mask1_path),
            'mask2_npy_sha256': sha256_file(mask2_path),
            'mask_npy_matches_png': _npy_matches_png(sid, m1, m2),
            'style1_name': spec['style1_name'],
            'style2_name': spec['style2_name'],
            'style1_sha256': STYLE_HASHES[spec['style1_name']],
            'style2_sha256': STYLE_HASHES[spec['style2_name']],
            'config_hash': CONFIG_HASH,
            'C1_METHOD_CONFIG_REFERENCE': (
                f'method configuration mirrors C1 (C1_RUN_ID={C1_RUN_ID}, '
                f'C1_CONFIG_HASH={C1_CONFIG_HASH}) (new qualitative sample, '
                f'not a C1 benchmark output)'
            ),
        }
        
    manifest_path = f'{BRUSHEDIT_BRIDGE_DIR}/manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    return manifest_path

## 12a. Method Metadata Schema

Structuring provenance records for every sample and baseline evaluated downstream in Section 13. Storing standard operational hyperparameters such as model checkpoints, sampling passes, and cumulative diffusion steps.Defining these fixed architectural attributes centrally ensures consistent logging across all qualitative runs.Covering the initial four pipelines evaluated directly in this environment: Method F, Sequential, Global, and Rectangular. The heavier architectures and BrushEdit export their respective schemas during Part 2 and the detached Colab notebook execution.

In [20]:
# Logging standardized architectural schemas across all local generation methods
METHOD_METADATA = {
    'Proposed -- Regional Cross-Attention (F)': {
        'method_group': 'primary',
        'backbone': CONFIG['sd_repo'], 'backbone_revision': None,
        'ip_adapter_checkpoint': CONFIG['ip_weight'], 'ip_adapter_scale': CONFIG['ip_scale'],
        'num_steps_per_pass': CONFIG['steps'], 'num_passes': 1,
        'total_diffusion_steps': CONFIG['steps'] * 1,
        'guidance_scale': CONFIG['guidance'], 'strength': CONFIG['strength'],
        'spatial_routing': 'regional (native ip_adapter_masks, one joint pass)',
        'postprocessing': None, 'compositing': 'diffusion-native (single inpainting pass)',
        'notes': 'proposed method',
    },
    'Sequential (naive two-pass)': {
        'method_group': 'primary',
        'backbone': CONFIG['sd_repo'], 'backbone_revision': None,
        'ip_adapter_checkpoint': CONFIG['ip_weight'], 'ip_adapter_scale': CONFIG['ip_scale'],
        'num_steps_per_pass': CONFIG['steps'], 'num_passes': 2,
        'total_diffusion_steps': CONFIG['steps'] * 2,
        'guidance_scale': CONFIG['guidance'], 'strength': CONFIG['strength'],
        'spatial_routing': 'none (two independent masked passes, one region at a time)',
        'postprocessing': None, 'compositing': 'diffusion-native (two sequential inpainting passes)',
        'notes': ("same checkpoint and masks as F, but uses two sequential passes; "
                  "the total diffusion-step budget is twice that of F"),
    },
    'IP-Adapter global (no routing)': {
        'method_group': 'primary',
        'backbone': CONFIG['sd_repo'], 'backbone_revision': None,
        'ip_adapter_checkpoint': CONFIG['ip_weight'], 'ip_adapter_scale': CONFIG['ip_scale'],
        'num_steps_per_pass': CONFIG['steps'], 'num_passes': 1,
        'total_diffusion_steps': CONFIG['steps'] * 1,
        'guidance_scale': CONFIG['guidance'], 'strength': CONFIG['strength'],
        'spatial_routing': 'none (global conditioning over a union mask, single joint prompt)',
        'postprocessing': None, 'compositing': 'diffusion-native (single inpainting pass)',
        'notes': ("uses the same IP-Adapter checkpoint as F; an earlier version "
                  "used ip-adapter-plus_sd15.bin and was replaced before the final comparison"),
    },
    'IP-Adapter + rect masks': {
        'method_group': 'primary',
        'backbone': CONFIG['sd_repo'], 'backbone_revision': None,
        'ip_adapter_checkpoint': CONFIG['ip_weight'], 'ip_adapter_scale': CONFIG['ip_scale'],
        'num_steps_per_pass': CONFIG['steps'], 'num_passes': 2,
        'total_diffusion_steps': CONFIG['steps'] * 2,
        'guidance_scale': CONFIG['guidance'], 'strength': CONFIG['strength'],
        'spatial_routing': ('rectangular (native ip_adapter_masks, geometric top/bottom '
                            'split, two sequential passes)'),
        'postprocessing': None, 'compositing': 'diffusion-native (two sequential inpainting passes)',
        'notes': ("uses rectangular masks and two sequential passes, so both the "
                  "mask shape and pass structure differ from F"),
    },
}

print('method metadata defined for:', list(METHOD_METADATA))

method metadata defined for: ['Proposed -- Regional Cross-Attention (F)', 'Sequential (naive two-pass)', 'IP-Adapter global (no routing)', 'IP-Adapter + rect masks']


## 13. Generation

## Generation -Part 1 (proposed F + light baselines only)

In [21]:
def _code_hash(*fns):
    return hashlib.sha256(''.join(inspect.getsource(f) for f in fns).encode()).hexdigest()[:16]


PROPOSED_CODE_HASH = _code_hash(generate_cross_attention_F, make_ip_masks)

METHOD_CODE_HASH = {'Proposed -- Regional Cross-Attention (F)': PROPOSED_CODE_HASH}
for _name, _fn in VALID_BASELINES.items():
    METHOD_CODE_HASH[_name] = _code_hash(_fn)
for _name, _entry in HEAVY_BASELINES.items():
    METHOD_CODE_HASH[_name] = _code_hash(_entry['fn'])


def output_paths(sample_id, method):
    slug = ''.join(c if c.isalnum() else '_' for c in method)
    d = f'{OUT_DIR}/{sample_id}/{slug}'
    os.makedirs(d, exist_ok=True)
    return f'{d}/output.png', f'{d}/provenance.json'


def load_cached(sample_id, method, expect):
    '''Cache validity is hash-based: content/mask/style/config/code hashes must all match
    what is currently loaded, or the cache is treated as invalid and regenerated. Never
    trusts an existing output.png merely because it exists.'''
    png, prov_path = output_paths(sample_id, method)
    if not (os.path.exists(png) and os.path.exists(prov_path)):
        return None
    try:
        with open(prov_path) as f:
            prov = json.load(f)
    except Exception:
        return None
    for k, v in expect.items():
        if prov.get(k) != v:
            return None
    img = Image.open(png).convert('RGB')
    img.load()
    return img


def save_output(sample_id, method, img, prov_extra):
    png, prov_path = output_paths(sample_id, method)
    img.save(png, 'PNG')
    # Static per-method fields (checkpoint, passes, step budget, ...) come from
    # METHOD_METADATA (Section 12a); per-sample fields come from prov_extra.
    rec = {'sample_id': sample_id, 'method': method, 'run_id': RUN_ID,
           'output_sha256': sha256_file(png), 'saved_at': datetime.now().isoformat(),
           **METHOD_METADATA.get(method, {}), **prov_extra}
    with open(prov_path, 'w') as f:
        json.dump(rec, f, indent=2)
    return rec


def provenance_expect(spec, sid, method):
    m1_h, m2_h = QUAL_MASK_META[sid]['h1'], QUAL_MASK_META[sid]['h2']
    return {
        'content_sha256': QUAL_CONTENT_HASH[sid],
        'mask1_sha256': m1_h, 'mask2_sha256': m2_h,
        'style1_sha256': STYLE_HASHES[spec['style1_name']],
        'style2_sha256': STYLE_HASHES[spec['style2_name']],
        'config_hash': CONFIG_HASH,
        'code_hash': METHOD_CODE_HASH.get(method),
        'seed': CONFIG['seed'], 'method': method,
    }


GENERATED = {sid: {} for sid in QUAL_SAMPLE_IDS}
GENERATION_FAILURES = []

require_gpu('generation')

print('=' * 70)
print('PART 1 -- proposed F + light baselines (per sample)')
print('Heavy baselines (NST/AdaIN/InstructPix2Pix/MultiDiffusion/ControlNet/SDXL/')
print('regional attention-bias) are NOT run here -- see Part 2. BrushEdit runs separately')
print('too (standalone notebook, e.g. on Colab) -- see Section 12 above.')
print('=' * 70)
for spec in QUAL_SAMPLES:
    sid = spec['id']
    content = QUAL_CONTENT[sid]
    m1, m2 = QUAL_MASKS[sid]
    s1, s2 = get_style_ref(spec['style1_name']), get_style_ref(spec['style2_name'])

    methods_to_run = {'Proposed -- Regional Cross-Attention (F)':
                       lambda: generate_cross_attention_F(content, m1, m2, s1, s2,
                                                           spec['prompt1'], spec['prompt2'],
                                                           seed=CONFIG['seed'])}
    for name, fn in VALID_BASELINES.items():
        methods_to_run[name] = (lambda fn=fn: fn(spec, content, m1, m2, s1, s2, seed=CONFIG['seed']))

    for method, fn in methods_to_run.items():
        expect = provenance_expect(spec, sid, method)
        cached = load_cached(sid, method, expect)
        if cached is not None:
            GENERATED[sid][method] = cached
            print(f'  {sid} / {method}: CACHED (hash-verified)')
            continue
        try:
            img = fn()
        except Exception as e:
            print(f'  {sid} / {method}: FAILED -- {type(e).__name__}: {e}')
            GENERATION_FAILURES.append({'sample_id': sid, 'method': method,
                                        'error': f'{type(e).__name__}: {e}'})
            GENERATED[sid][method] = None
            continue
        prov = save_output(sid, method, img, expect)
        GENERATED[sid][method] = img
        print(f'  {sid} / {method}: generated, output_sha256={prov["output_sha256"][:12]}')

if GENERATION_FAILURES:
    print(f'\n{len(GENERATION_FAILURES)} generation failure(s) in Part 1:')
    for f in GENERATION_FAILURES:
        print(f"  {f['sample_id']} / {f['method']}: {f['error']}")

print(f'\nPart 1 complete. Everything above is now saved under {OUT_DIR!r} on Drive.')


PART 1 -- proposed F + light baselines (per sample)
Heavy baselines (NST/AdaIN/InstructPix2Pix/MultiDiffusion/ControlNet/SDXL/
regional attention-bias) are NOT run here -- see Part 2. BrushEdit runs separately
too (standalone notebook, e.g. on Colab) -- see Section 12 above.


  0%|          | 0/22 [00:00<?, ?it/s]

  QUAL_01 / Proposed -- Regional Cross-Attention (F): generated, output_sha256=14330bee1bc1


  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

...

  0%|          | 0/22 [00:00<?, ?it/s]

  QUAL_03 / IP-Adapter global (no routing): generated, output_sha256=a79f761ba942


  0%|          | 0/22 [00:00<?, ?it/s]

  0%|          | 0/22 [00:00<?, ?it/s]

  QUAL_03 / IP-Adapter + rect masks: generated, output_sha256=31f65816ecc6

Part 1 complete. Everything above is now saved under '/mnt/vurm/homes/homes/rkr44/baselines_work/outputs/QUALRUN_20260919T082948Z' on Drive.
Full run tree (masks, style refs, content, weights cache): '/mnt/vurm/homes/homes/rkr44/baselines_work'

NEXT: open Part 2 in a fresh runtime. It re-runs Sections 1-11 (fast -- everything
above is Drive-cached, so this is a cache-hit pass, not a re-download/re-detect
pass) to rebuild QUAL_CONTENT/QUAL_MASKS/STYLE_REFS/MODELS in memory, then picks up
Part 1's cached outputs automatically before running the heavy baselines, then
provenance/visualisation/verification. Nothing from Part 1 needs to be re-generated.

If this run only regenerated the corrected Global baseline (RUN_ID_OVERRIDE set in
Section 0a), paste the SAME RUN_ID into Part 2 as before -- Part 2 will reattach the
unchanged F/Sequential/Rectangular outputs and pick up the corrected Global output.


In [29]:
print(f"RUN_ID_OVERRIDE = {RUN_ID!r}")

RUN_ID_OVERRIDE = 'QUALRUN_20260919T082948Z'


## 15. Export

Bundling artifacts into a self-contained archive for downstream comparison in Notebook 4[cite: 1]. The package bundles `manifest.json` (one standardized provenance record per sample/method pairing), `configs.json` (the locked generation parameters), a brief `README.md`, and all generated images into `PART1_BASELINE_EXPORT.zip`.The pipeline pulls strictly from the active `RUN_ID` directory within `OUT_DIR`, preventing superseded experiments—such as previous exploratory global runs—from being included in the final benchmark archive[cite: 1].

In [33]:
import shutil

EXPORT_DIR = f'{WORK}/PART1_BASELINE_EXPORT'
EXPORT_OUTPUTS_DIR = f'{EXPORT_DIR}/outputs'
EXPORT_ASSETS_DIR = f'{EXPORT_DIR}/assets'
os.makedirs(EXPORT_OUTPUTS_DIR, exist_ok=True)
os.makedirs(EXPORT_ASSETS_DIR, exist_ok=True)
assets_manifest = {}
for sid in QUAL_SAMPLE_IDS:
    spec = next(s for s in QUAL_SAMPLES if s['id'] == sid)
    d = f'{EXPORT_ASSETS_DIR}/{sid}'
    os.makedirs(d, exist_ok=True)

    QUAL_CONTENT[sid].save(f'{d}/content.png')
    get_style_ref(spec['style1_name']).save(f'{d}/style_reference_1.png')
    get_style_ref(spec['style2_name']).save(f'{d}/style_reference_2.png')
    m1, m2 = QUAL_MASKS[sid]
    Image.fromarray(m1).save(f'{d}/mask_1.png')
    Image.fromarray(m2).save(f'{d}/mask_2.png')

    assets_manifest[sid] = {
        'content': f'assets/{sid}/content.png',
        'style_reference_1': f'assets/{sid}/style_reference_1.png',
        'style_reference_2': f'assets/{sid}/style_reference_2.png',
        'mask_1': f'assets/{sid}/mask_1.png',
        'mask_2': f'assets/{sid}/mask_2.png',
        'style_reference_1_name': spec['style1_name'],
        'style_reference_2_name': spec['style2_name'],
    }

with open(f'{EXPORT_DIR}/assets.json', 'w') as f:
    json.dump(assets_manifest, f, indent=2)

manifest_rows = []
for sid in QUAL_SAMPLE_IDS:
    spec = next(s for s in QUAL_SAMPLES if s['id'] == sid)
    meta = QUAL_MASK_META[sid]
    for method, img in GENERATED[sid].items():
        src_png, prov_path = output_paths(sid, method)
        status = 'ok' if img is not None else 'failed'
        prov = {}
        if os.path.exists(prov_path):
            with open(prov_path) as f:
                prov = json.load(f)

        dst_dir = f'{EXPORT_OUTPUTS_DIR}/{sid}'
        os.makedirs(dst_dir, exist_ok=True)
        slug = ''.join(c if c.isalnum() else '_' for c in method)
        dst_png = f'{dst_dir}/{slug}.png'
        if status == 'ok':
            shutil.copy(src_png, dst_png)
            output_path_rel = f'outputs/{sid}/{slug}.png'
        else:
            output_path_rel = None

        row = {
            'method_name': method,
            'method_group': prov.get('method_group', METHOD_METADATA.get(method, {}).get('method_group')),
            'sample_id': sid, 'content_id': sid,
            'style_reference_1': spec['style1_name'], 'style_reference_2': spec['style2_name'],
            'mask_1': meta['h1'], 'mask_2': meta['h2'],
            'prompt': f"{spec['prompt1']} | {spec['prompt2']}",
            'seed': CONFIG['seed'],
            'backbone': prov.get('backbone'), 'backbone_revision': prov.get('backbone_revision'),
            'ip_adapter_checkpoint': prov.get('ip_adapter_checkpoint'),
            'ip_adapter_scale': prov.get('ip_adapter_scale'),
            'num_steps_per_pass': prov.get('num_steps_per_pass'),
            'num_passes': prov.get('num_passes'),
            'total_diffusion_steps': prov.get('total_diffusion_steps'),
            'guidance_scale': prov.get('guidance_scale'), 'strength': prov.get('strength'),
            'spatial_routing': prov.get('spatial_routing'),
            'postprocessing': prov.get('postprocessing'),
            'compositing': prov.get('compositing'),
            'output_path': output_path_rel,
            'output_sha256': prov.get('output_sha256'),
            'status': status,
            'notes': METHOD_METADATA.get(method, {}).get('notes'),
        }
        manifest_rows.append(row)

with open(f'{EXPORT_DIR}/manifest.json', 'w') as f:
    json.dump(manifest_rows, f, indent=2)

with open(f'{EXPORT_DIR}/configs.json', 'w') as f:
    json.dump({'config': CONFIG, 'config_hash': CONFIG_HASH, 'run_id': RUN_ID,
               'c1_run_id': C1_RUN_ID, 'c1_config_hash': C1_CONFIG_HASH,
               'c1_reference_hash_match': 'NOT VERIFIED -- C1_VERIFICATION was not '
                                          'loadable in this environment (Section 2)'},
              f, indent=2)

# Store the same information in one nested package for Notebook 4.
package = {
    'package_metadata': {
        'baseline_export_version': '1.0',
        'source_notebook': 'FINAL_DISSERTATION_ROUTING_BASELINES_PART1',
        'run_id': RUN_ID,
        'generated_at': datetime.now().isoformat(),
    },
    'methods': sorted(set(r['method_name'] for r in manifest_rows)),
    'samples': QUAL_SAMPLE_IDS,
    'configs': {'config': CONFIG, 'config_hash': CONFIG_HASH},
    'metrics': None,
    'assets': assets_manifest,
    'outputs': manifest_rows,
}
with open(f'{EXPORT_DIR}/export_package.json', 'w') as f:
    json.dump(package, f, indent=2)
## Methods included
{chr(10).join('- ' + m for m in sorted(set(r['method_name'] for r in manifest_rows)))}

## Samples included
{chr(10).join('- ' + s for s in QUAL_SAMPLE_IDS)}
with open(f'{EXPORT_DIR}/README.md', 'w') as f:
    f.write(readme)

zip_path = shutil.make_archive(f'{WORK}/PART1_BASELINE_EXPORT', 'zip', root_dir=EXPORT_DIR)
print(f'PART1_BASELINE_EXPORT.zip written to {zip_path}')
print(f'  methods: {sorted(set(r["method_name"] for r in manifest_rows))}')
print(f'  rows: {len(manifest_rows)}  (status ok: {sum(r["status"]=="ok" for r in manifest_rows)}, '
      f'failed: {sum(r["status"]=="failed" for r in manifest_rows)})')

# Create the separate BrushEdit input package.
bridge_zip = f'{WORK}/brushedit_bridge_{RUN_ID}'
shutil.make_archive(bridge_zip, 'zip', root_dir=BRUSHEDIT_BRIDGE_DIR)
print(f'BrushEdit bridge: {bridge_zip}.zip')

try:
    from google.colab import files
    files.download(zip_path)
    files.download(f'{bridge_zip}.zip')
except ImportError:
    print('Not on Colab')

PART1_BASELINE_EXPORT.zip written to /mnt/vurm/homes/homes/rkr44/baselines_work/PART1_BASELINE_EXPORT.zip
  methods: ['IP-Adapter + rect masks', 'IP-Adapter global (no routing)', 'Proposed -- Regional Cross-Attention (F)', 'Sequential (naive two-pass)']
  rows: 12  (status ok: 12, failed: 0)
BrushEdit bridge: /mnt/vurm/homes/homes/rkr44/baselines_work/brushedit_bridge_QUALRUN_20260919T082948Z.zip
Not on Colab
